# CRDC 2017-18: school-level cleaning (Task #3.2, Person B)

This notebook turns the five 2017-18 CRDC module files into the team's shared school table.
It follows the team cleaning guide (`cleaning_feature_playbook.md`, PR #20): the shared steps
in section 1, the 50-column output in section 3, the formulas in section 4, and the Person B notes in section 5.

**What it produces** (in `data/processed/201718/`):

| File | Contents |
|---|---|
| `schools_201718.csv` | every roster school (97,632 rows), exactly the 50 columns of playbook section 3.1 |
| `schools_201718_analysis.csv` | the same table filtered to `flag_analysis_ready == True` |
| `quality_detail_201718.csv` | one row per data problem found (reserved codes, odd IDs, outliers, ...) |

**What it does not do** (left for later team decisions): no imputation, no filling unknowns with 0,
no scaling, no capping, no row deletion, no high-seclusion threshold.

**How to run:** *Restart Kernel and Run All* (or, from the project root,
`jupyter-execute --inplace notebooks/clean_201718.ipynb`). On Windows, set `PYTHONUTF8=1` before running `jupyter-execute` (PowerShell: `$env:PYTHONUTF8 = "1"`), or it garbles any non-ASCII characters in this notebook.
The only line you might need to change is
`PROJECT_ROOT` in the configuration cell. Running it twice gives byte-identical CSVs. Step 12 reloads
the written files and checks them against the playbook section 6 list, with hand-checked examples.

Field-by-field sources and manual page references are in `docs/cleaning/mapping_201718.csv`.
Abbreviations used below: **MAN** = 2017-18 CRDC Public Data File Manual, **TD** = 2017-18 CRDC
Technical Document, **ML** = 2017-18 Master List of CRDC Definitions.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.3.3 | numpy 2.2.6


## Configuration
All input and output paths are built from one `PROJECT_ROOT`: the folder above `notebooks/` when the
kernel starts there (the Jupyter / VS Code default), otherwise the current folder.

In [2]:
# Works when the kernel starts in notebooks/ (Jupyter, VS Code) or in the project root
# (e.g. `jupyter-execute` run from the root). If your setup differs, change only this line.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()

YEAR = "2017-18"      # label written in the `year` column
SUFFIX = "201718"     # filename suffix

RAW_DIR = PROJECT_ROOT / "data" / "raw_extracted" / "2017-18"
INPUTS = {
    "C": RAW_DIR / "School Characteristics.csv",
    "E": RAW_DIR / "Enrollment.csv",
    "R": RAW_DIR / "Restraint and Seclusion.csv",
    "S": RAW_DIR / "School Support.csv",
    "L": RAW_DIR / "LEA Characteristics.csv",
}
OUT_DIR = PROJECT_ROOT / "data" / "processed" / SUFFIX
OUT_FULL = OUT_DIR / f"schools_{SUFFIX}.csv"
OUT_ANALYSIS = OUT_DIR / f"schools_{SUFFIX}_analysis.csv"
OUT_QUALITY = OUT_DIR / f"quality_detail_{SUFFIX}.csv"

In [3]:
def rel(path):
    # Path relative to the project root, so printed output never shows a local absolute path.
    return path.relative_to(PROJECT_ROOT).as_posix()


def vc(series):
    # value_counts as a plain {value: int} dict (readable when printed).
    return {(k.item() if hasattr(k, "item") else k): int(v) for k, v in series.value_counts(dropna=False).items()}

assert RAW_DIR.is_dir(), (
    f"Raw data folder not found: {RAW_DIR}. PROJECT_ROOT must be the project folder that contains "
    "data/raw_extracted/2017-18 (run from notebooks/ or the project root, or edit PROJECT_ROOT).")
missing_inputs = [str(p) for p in INPUTS.values() if not p.exists()]
assert not missing_inputs, f"Input files not found; check PROJECT_ROOT. Missing: {missing_inputs}"
print("project root folder:", PROJECT_ROOT.name)
for key, path in INPUTS.items():
    print(f"  {key}: {rel(path)}")
print("outputs go to:", rel(OUT_DIR))

project root folder: Data_Cleaning_2017-2018
  C: data/raw_extracted/2017-18/School Characteristics.csv
  E: data/raw_extracted/2017-18/Enrollment.csv
  R: data/raw_extracted/2017-18/Restraint and Seclusion.csv
  S: data/raw_extracted/2017-18/School Support.csv
  L: data/raw_extracted/2017-18/LEA Characteristics.csv
outputs go to: data/processed/201718


### Fields used from each module
These are the raw headers checked in Phase 1 (`docs/cleaning/profile_201718.txt`). Grade headers
use two digits (`SCH_GRADE_G01` ... `G12`), not `G1` ... `G12` as the playbook writes them.
`SCH_GRADE_PS` and `SCH_GRADE_UG` are read only to explain `grade_band = "other"` (raw text, not
cleaned or logged: they feed no output column); the band itself uses KG and grades 1-12 only.

In [4]:
GRADE_COLS = ["SCH_GRADE_KG"] + [f"SCH_GRADE_G{g:02d}" for g in range(1, 13)]
STATUS_COLS = {
    "SCH_STATUS_SPED": "sch_status_sped",
    "SCH_STATUS_CHARTER": "sch_status_charter",
    "SCH_STATUS_ALT": "sch_status_alt",
    "SCH_STATUS_MAGNET": "sch_status_magnet",
}
SECL_COLS = {
    "SCH_RSINSTANCES_SECL_WODIS": "seclusion_instances_wodis",
    "SCH_RSINSTANCES_SECL_IDEA": "seclusion_instances_idea",
    "SCH_RSINSTANCES_SECL_504": "seclusion_instances_504",
}
ENR_COLS = {  # raw -> intermediate name (M/F parts are not output columns)
    "TOT_ENR_M": "tot_enr_m", "TOT_ENR_F": "tot_enr_f",
    "SCH_ENR_IDEA_M": "idea_m", "SCH_ENR_IDEA_F": "idea_f",
    "SCH_ENR_504_M": "s504_m", "SCH_ENR_504_F": "s504_f",
    "SCH_ENR_LEP_M": "el_m", "SCH_ENR_LEP_F": "el_f",
}
FTE_COLS = {
    "SCH_FTETEACH_TOT": "fte_teachers",
    "SCH_FTECOUNSELORS": "fte_counselors",
    "SCH_FTESERVICES_NUR": "fte_nurses",
    "SCH_FTESERVICES_PSY": "fte_psych",
    "SCH_FTESERVICES_SOC": "fte_social",
    "SCH_FTESECURITY_LEO": "fte_leo",
    "SCH_FTESECURITY_GUA": "fte_guards",
}
LEA_COUNT_COLS = {"LEA_ENR": "lea_enrollment", "LEA_SCHOOLS": "lea_schools"}

NEEDED = {
    "C": ["LEA_STATE", "LEAID", "LEA_NAME", "SCHID", "SCH_NAME", "COMBOKEY", "JJ",
          "SCH_GRADE_PS", *GRADE_COLS, "SCH_GRADE_UG", *STATUS_COLS],
    "E": ["COMBOKEY", *ENR_COLS],
    # R's SCHID is read only to show it is wrong; it is dropped before the join.
    "R": ["COMBOKEY", "SCHID", *SECL_COLS],
    "S": ["COMBOKEY", *FTE_COLS],
    "L": ["LEAID", "LEA_STATE", "LEA_NAME", *LEA_COUNT_COLS],
}

# What Phase 1 found (compare, then stop if different).
EXPECTED_ENCODING = {"C": "cp1252", "E": "utf-8-sig", "R": "utf-8-sig", "S": "cp1252", "L": "cp1252"}
EXPECTED_ROWS = {"C": 97_632, "E": 97_632, "R": 97_632, "S": 97_632, "L": 17_604}
EXPECTED_SHA256 = {
    "C": "4cbe0d51cc9c0c32bd9c57c5f608aa93762a67282698739ff98a276e7470d5d0",
    "E": "7a62a3b6634e1b2c375c4af1866317b6949bb3612bf912ec31f638dfe5adab76",
    "R": "780c2ef2d209ae6d81207d311b674a7ee4d0053b54cf8452d06718403053084f",
    "S": "75fddde40c18f1220921ce71fd2532e2760aa90ae38bf681f198d1987d8814aa",
    "L": "87afd6ce7cff48bdda1f5bac1f0cf04136d4f194ec2d021c2918aa869b69c909",
}

### Quality-detail log
Every problem found is written to one long table (`quality_detail_201718.csv`, playbook section 3.3), one
issue type per row: `reserved_code`, `blank_cell`, `unparseable_value`, `non_finite_value`,
`non_integer_count`, `unexpected_category`, `missing_module_row`, `missing_lea_row`, `non_digit_id`,
`subgroup_above_total`, `fte_outlier`, `name_disagreement`. `source_row` is the parsed record
number in the source file (first data row = 1), not a physical line number.

In [5]:
QD_COLUMNS = ["year", "source_file", "source_row", "combokey", "leaid", "raw_column",
              "raw_value", "issue", "action", "cleaned_value", "evidence"]
quality_parts = []   # one small DataFrame per log call; concatenated before writing


def log_issues(frame, mask, *, source_file, raw_column, raw_value, issue, action,
               cleaned_value="", evidence="", row_col="_row"):
    # Append one quality-detail row per True value of `mask`. Returns the number logged.
    mask = mask.fillna(False).astype(bool)
    n = int(mask.sum())
    if n == 0:
        return 0

    def pick(value):
        return value[mask].to_numpy() if isinstance(value, pd.Series) else value

    sub = frame.loc[mask]
    quality_parts.append(pd.DataFrame({
        "year": YEAR,
        "source_file": source_file,
        "source_row": sub[row_col].to_numpy(),
        "combokey": sub["combokey"].to_numpy() if "combokey" in sub else "",
        "leaid": sub["leaid"].to_numpy(),
        "raw_column": raw_column,
        "raw_value": pick(raw_value),
        "issue": issue,
        "action": action,
        "cleaned_value": pick(cleaned_value),
        "evidence": pick(evidence),
    }, columns=QD_COLUMNS))
    return n


def compare(label, actual, expected):
    # Print actual next to an expected value from Phases 0-2. Differences are reported, not forced.
    flag = "match" if actual == expected else "DIFFERS"
    print(f"  {label:<52} expected {expected:>8,}   actual {actual:>8,}   {flag}")


STEP_ROWS = []   # (step, roster rows) after each step

## Step 1 - Read the files (playbook section 1, step 1)
Each file is read with every field as text, so nothing is silently turned into a number or into
NaN (`keep_default_na=False, na_filter=False`); literal text such as "NA" stays visible and can be
logged. Encoding: try `utf-8-sig`, then `cp1252` (three of the five files need `cp1252`, more than the
playbook's section 2.2 expects). Whitespace is stripped and completely empty rows are counted.

In [6]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def read_module(path, columns):
    # Read only `columns`, all as strings. Returns (DataFrame, encoding used).
    for encoding in ("utf-8-sig", "cp1252"):
        try:
            header = pd.read_csv(path, nrows=0, encoding=encoding).columns
            missing = [c for c in columns if c not in header]
            if missing:
                raise KeyError(f"{path.name}: expected columns missing: {missing}")
            df = pd.read_csv(path, dtype=str, keep_default_na=False, na_filter=False,
                             encoding=encoding, usecols=columns)
            return df[columns], encoding
        except UnicodeDecodeError:
            continue
    raise ValueError(f"{path.name}: could not decode as utf-8-sig or cp1252")


def strip_all(df):
    # Strip whitespace in every column; return the stripped frame and cells changed per column.
    stripped = df.apply(lambda s: s.str.strip())
    changed = {c: int((stripped[c] != df[c]).sum()) for c in df.columns}
    return stripped, {c: n for c, n in changed.items() if n}

In [7]:
raw = {}
READ_LOG = {}
for key, path in INPUTS.items():
    digest = sha256(path)
    df, encoding = read_module(path, NEEDED[key])
    df, stripped = strip_all(df)
    df["_row"] = np.arange(1, len(df) + 1)   # parsed record number, first data row = 1
    raw[key] = df
    READ_LOG[key] = {
        "file": rel(path), "encoding": encoding, "rows": len(df), "cols_read": len(NEEDED[key]),
        "empty_rows": int((df[NEEDED[key]] == "").all(axis=1).sum()),
        "cells_stripped": stripped, "sha256": digest,
    }
pd.DataFrame(READ_LOG).T[["file", "encoding", "rows", "cols_read", "empty_rows", "cells_stripped"]]

,file,encoding,rows,cols_read,empty_rows,cells_stripped
C,data/raw_extracted/2017-18/School Characterist...,cp1252,97632,26,0,{}
E,data/raw_extracted/2017-18/Enrollment.csv,utf-8-sig,97632,9,0,{}
R,data/raw_extracted/2017-18/Restraint and Seclu...,utf-8-sig,97632,5,0,{}
S,data/raw_extracted/2017-18/School Support.csv,cp1252,97632,8,0,{}
L,data/raw_extracted/2017-18/LEA Characteristics...,cp1252,17604,5,0,{}


In [8]:
for key, info in READ_LOG.items():
    assert info["sha256"] == EXPECTED_SHA256[key], f"{info['file']}: SHA-256 differs from the Phase 1 record"
    assert info["encoding"] == EXPECTED_ENCODING[key], f"{info['file']}: encoding {info['encoding']} != Phase 1"
    assert info["rows"] == EXPECTED_ROWS[key], f"{info['file']}: {info['rows']:,} rows != {EXPECTED_ROWS[key]:,}"
    assert info["empty_rows"] == 0, f"{info['file']}: {info['empty_rows']} completely empty rows"
N_ROSTER = len(raw["C"])
STEP_ROWS.append(("1 read (School Characteristics roster)", N_ROSTER))
print("All 5 files match the Phase 1 hashes, encodings and row counts; no empty rows.")
print(f"Roster: {N_ROSTER:,} schools; LEA file: {len(raw['L']):,} districts.")

All 5 files match the Phase 1 hashes, encodings and row counts; no empty rows.
Roster: 97,632 schools; LEA file: 17,604 districts.


## Step 2 - IDs (playbook section 1, step 2, section 5 Person B)
IDs are checked *before* any padding: no blanks, digits only, not too long. `LEAID` pads to 7,
`SCHID` to 5 and `COMBOKEY` to 12, and `COMBOKEY` must equal `LEAID + SCHID`. 2017-18 has 510
schools (452 districts) with official letter IDs such as `06CC001` (California charters) and
`24SOP01` (state-run programs). Per Phase 1 decision 1 they are accepted as `^[0-9A-Z]{7}$`, kept
unchanged (not padded) and logged as `non_digit_id`. `SCHID` must still be 5 digits.

In [9]:
def classify_leaid(leaid):
    # 'digit' = 1-7 digits (padded to 7); 'letter' = official 7-char ID with letters; else 'invalid'.
    digit = leaid.str.fullmatch(r"\d{1,7}")
    letter = ~digit & leaid.str.fullmatch(r"[0-9A-Z]{7}")
    return pd.Series(np.select([digit, letter], ["digit", "letter"], "invalid"), index=leaid.index)


def pad_leaid(leaid, kind):
    return leaid.where(kind != "digit", leaid.str.zfill(7))


def check_combokey(combokey):
    # 'digit' = 1-12 digits (padded to 12); 'letter' = 12 chars of [0-9A-Z]; else 'invalid'.
    digit = combokey.str.fullmatch(r"\d{1,12}")
    letter = ~digit & combokey.str.fullmatch(r"[0-9A-Z]{12}")
    kind = pd.Series(np.select([digit, letter], ["digit", "letter"], "invalid"), index=combokey.index)
    padded = combokey.where(kind != "digit", combokey.str.zfill(12))
    return kind, padded


def id_summary(name, s):
    return {"column": name, "blank": int((s == "").sum()), "non_digit": int((~s.str.fullmatch(r"\d*")).sum()),
            "lengths": vc(s.str.len())}

In [10]:
c = raw["C"].copy()
print("Before padding (School Characteristics):")
for col, max_len in [("LEAID", 7), ("SCHID", 5), ("COMBOKEY", 12)]:
    info = id_summary(col, c[col])
    info["too_long"] = int((c[col].str.len() > max_len).sum())
    print("  ", info)

c_lea_kind = classify_leaid(c["LEAID"])
c_sch_ok = c["SCHID"].str.fullmatch(r"\d{1,5}")
c_combo_kind, c_combo = check_combokey(c["COMBOKEY"])
c["leaid"] = pad_leaid(c["LEAID"], c_lea_kind)
c["schid"] = c["SCHID"].where(~c_sch_ok, c["SCHID"].str.zfill(5))
c["combokey"] = c_combo
print("LEAID kinds:", vc(c_lea_kind), "| COMBOKEY kinds:", vc(c_combo_kind))
print("values changed by padding: LEAID", int((c["leaid"] != c["LEAID"]).sum()),
      "| SCHID", int((c["schid"] != c["SCHID"]).sum()), "| COMBOKEY", int((c["combokey"] != c["COMBOKEY"]).sum()))

Before padding (School Characteristics):
   {'column': 'LEAID', 'blank': 0, 'non_digit': 510, 'lengths': {7: 97632}, 'too_long': 0}
   {'column': 'SCHID', 'blank': 0, 'non_digit': 0, 'lengths': {5: 97632}, 'too_long': 0}
   {'column': 'COMBOKEY', 'blank': 0, 'non_digit': 510, 'lengths': {12: 97632}, 'too_long': 0}


LEAID kinds: {'digit': 97122, 'letter': 510} | COMBOKEY kinds: {'digit': 97122, 'letter': 510}
values changed by padding: LEAID 0 | SCHID 0 | COMBOKEY 0


In [11]:
assert (c_lea_kind != "invalid").all(), f"Invalid LEAIDs: {c.loc[c_lea_kind == 'invalid', 'LEAID'].head().tolist()}"
assert c_sch_ok.all(), f"Invalid SCHIDs: {c.loc[~c_sch_ok, 'SCHID'].head().tolist()}"
assert (c_combo_kind != "invalid").all(), "Invalid COMBOKEYs in School Characteristics"
assert (c["combokey"] == c["leaid"] + c["schid"]).all(), "COMBOKEY != LEAID + SCHID on some roster rows"
assert ((c_lea_kind == "letter") == (c_combo_kind == "letter")).all(), "letter LEAID and letter COMBOKEY disagree"
assert not c["combokey"].duplicated().any(), "Duplicate COMBOKEY in the roster"
assert c["leaid"].str.len().eq(7).all() and c["schid"].str.len().eq(5).all() and c["combokey"].str.len().eq(12).all()

letter_ids = c_lea_kind == "letter"
n_logged = log_issues(
    c, letter_ids, source_file="School Characteristics.csv", raw_column="LEAID", raw_value=c["LEAID"],
    issue="non_digit_id", action="retained", cleaned_value=c["leaid"],
    evidence="Official CRDC LEAID with letters (06CC### CA charter LEA or ##SOP## state-operated "
             "program); accepted as ^[0-9A-Z]{7}$ and not padded (Phase 1 decision 1). "
             "COMBOKEY = LEAID + SCHID verified.")
compare("roster rows with a letter LEAID (non_digit_id logged)", n_logged, 510)
compare("distinct letter LEAIDs in the roster", c.loc[letter_ids, "leaid"].nunique(), 452)
print("Letter-ID shapes:", vc(c.loc[letter_ids, "LEAID"].str.replace(r"[0-9]", "9", regex=True)
                             .str.replace(r"[A-Z]", "A", regex=True)))

  roster rows with a letter LEAID (non_digit_id logged) expected      510   actual      510   match
  distinct letter LEAIDs in the roster                 expected      452   actual      452   match
Letter-ID shapes: {'99AA999': 482, '99AAA99': 28}


The other modules are joined on `COMBOKEY` only (and the LEA file on `LEAID`), so here their keys
are checked for format and duplicates. Restraint and Seclusion's own `SCHID` is wrong on almost
every row (Phase 1: 97,630 of 97,632); it is checked here only to confirm that, and is never used.

In [12]:
roster_leaid = dict(zip(c["combokey"], c["leaid"]))
mods = {}
for key in ["E", "R", "S"]:
    df = raw[key].copy()
    kind, df["combokey"] = check_combokey(df["COMBOKEY"])
    assert (kind != "invalid").all(), f"{INPUTS[key].name}: invalid COMBOKEYs {df.loc[kind == 'invalid', 'COMBOKEY'].head().tolist()}"
    assert not df["combokey"].duplicated().any(), f"{INPUTS[key].name}: duplicate COMBOKEYs"
    df["leaid"] = df["combokey"].map(roster_leaid).fillna("")   # leaid for the quality log
    mods[key] = df
    print(f"{INPUTS[key].name}: {len(df):,} rows, COMBOKEY kinds {vc(kind)}, 0 duplicates")

r_schid_check = mods["R"]["combokey"].str[-5:] != mods["R"]["SCHID"]
compare("Restraint and Seclusion rows where SCHID != COMBOKEY[-5:]", int(r_schid_check.sum()), 97_630)

lea = raw["L"].copy()
l_kind = classify_leaid(lea["LEAID"])
assert (l_kind != "invalid").all(), f"LEA file: invalid LEAIDs {lea.loc[l_kind == 'invalid', 'LEAID'].head().tolist()}"
lea["leaid"] = pad_leaid(lea["LEAID"], l_kind)
assert not lea["leaid"].duplicated().any(), "LEA file: duplicate LEAIDs"
print(f"LEA Characteristics.csv: {len(lea):,} rows, LEAID kinds {vc(l_kind)}, 0 duplicates")
STEP_ROWS.append(("2 IDs validated", len(c)))

Enrollment.csv: 97,632 rows, COMBOKEY kinds {'digit': 97122, 'letter': 510}, 0 duplicates

Restraint and Seclusion.csv: 97,632 rows, COMBOKEY kinds {'digit': 97122, 'letter': 510}, 0 duplicates
School Support.csv: 97,632 rows, COMBOKEY kinds {'digit': 97122, 'letter': 510}, 0 duplicates
  Restraint and Seclusion rows where SCHID != COMBOKEY[-5:] expected   97,630   actual   97,630   match


LEA Characteristics.csv: 17,604 rows, LEAID kinds {'digit': 17152, 'letter': 452}, 0 duplicates


## Step 3 - Counts, FTEs and reserved codes (playbook section 1, step 3)
Each count/FTE text value becomes a number. Every negative number is a reserved code (MAN section 5.1.1,
Table 2, p.12) and becomes NA; the original text (e.g. `-9.00`) goes to the quality log. Codes are
detected numerically, because FTE codes are written `-5.00`, `-6.00`, `-9.00`. Blanks, text that
won't parse, and infinite values also become NA and are logged separately. Reported zeros stay 0;
nothing is filled in. Counts must be whole numbers; fractional FTEs are valid.

In [13]:
MAN_TABLE2 = "2017-18 CRDC Public Data File Manual sec. 5.1.1 Table 2 p.12"
RESERVED_MEANING = {
    -3: "Skip Logic Failure", -5: "Action Plan", -6: "Force Certified",
    -8: "EDFacts Missing", -9: "Not Applicable/Skipped", -11: "Suppressed",
}
NYC_LEAID = "3620580"
NYC_NOTE = ("; New York City: 2017-18 CRDC Technical Document sec. 7.1 p.11 says these were coded "
            "force certified but should have been action-plan codes")


def reserved_evidence(codes):
    def text(code):
        meaning = RESERVED_MEANING.get(int(code))
        return f"{int(code)} = {meaning} ({MAN_TABLE2})" if meaning else f"{code:g}: code not listed in {MAN_TABLE2} (unresolved)"
    return codes.map(text)


def clean_numeric(frame, raw_col, *, source_file, kind, row_col="_row", extra_evidence=None):
    # Return a Float64 Series: valid values >= 0 kept, everything else NA (and logged).
    text = frame[raw_col]
    num = pd.to_numeric(text, errors="coerce")
    blank = text.eq("")
    unparseable = num.isna() & ~blank
    non_finite = np.isinf(num)
    reserved = num < 0
    common = dict(source_file=source_file, raw_column=raw_col, raw_value=text, row_col=row_col)

    evidence = pd.Series("", index=text.index)
    evidence[reserved] = reserved_evidence(num[reserved])
    if extra_evidence is not None:
        evidence = evidence + extra_evidence
    counts = {
        "reserved_code": log_issues(frame, reserved, issue="reserved_code", action="set_na",
                                    evidence=evidence, **common),
        "blank_cell": log_issues(frame, blank, issue="blank_cell", action="set_na", **common),
        "unparseable_value": log_issues(frame, unparseable, issue="unparseable_value", action="set_na", **common),
        "non_finite_value": log_issues(frame, non_finite, issue="non_finite_value", action="set_na", **common),
    }
    clean = num.where(~(reserved | non_finite)).astype("Float64")
    if kind == "count":
        fraction = clean.notna() & (clean % 1 != 0)
        counts["non_integer_count"] = log_issues(
            frame, fraction, issue="non_integer_count", action="retained_pending_review",
            cleaned_value=clean.astype(str), evidence="Counts must be whole numbers (playbook sec. 1 step 3)", **common)
    codes = vc(text[reserved])
    return clean, counts, codes

In [14]:
NUMERIC_SUMMARY = []


def clean_module(frame, columns, *, source_file, kind, extra_evidence=None):
    # Clean several raw columns; returns a DataFrame of cleaned values named by `columns` (raw -> new).
    out = pd.DataFrame(index=frame.index)
    for raw_col, new_col in columns.items():
        extra = extra_evidence(frame, raw_col) if extra_evidence else None
        out[new_col], counts, codes = clean_numeric(frame, raw_col, source_file=source_file,
                                                    kind=kind, extra_evidence=extra)
        valid = out[new_col].dropna()
        NUMERIC_SUMMARY.append({"source_column": raw_col, "valid": len(valid), "NA": int(out[new_col].isna().sum()),
                                "zero": int((valid == 0).sum()), "min": valid.min(), "max": valid.max(),
                                "reserved_codes": codes, **{k: v for k, v in counts.items() if k != "reserved_code"}})
    return out


def nyc_security_note(frame, raw_col):
    # Extra evidence text for New York City's -6 security FTE codes (TD sec. 7.1).
    is_nyc_code = (frame["leaid"] == NYC_LEAID) & (pd.to_numeric(frame[raw_col], errors="coerce") == -6)
    return pd.Series(np.where(is_nyc_code & raw_col.startswith("SCH_FTESECURITY"), NYC_NOTE, ""), index=frame.index)


e_clean = clean_module(mods["E"], ENR_COLS, source_file="Enrollment.csv", kind="count")
r_clean = clean_module(mods["R"], SECL_COLS, source_file="Restraint and Seclusion.csv", kind="count")
s_clean = clean_module(mods["S"], FTE_COLS, source_file="School Support.csv", kind="fte",
                       extra_evidence=nyc_security_note)
l_clean = clean_module(lea, LEA_COUNT_COLS, source_file="LEA Characteristics.csv", kind="count")

pd.set_option("display.width", 200, "display.max_colwidth", 80)
pd.DataFrame(NUMERIC_SUMMARY).set_index("source_column")

,valid,NA,zero,min,max,reserved_codes,blank_cell,unparseable_value,non_finite_value,non_integer_count
source_column,,,,,,,,,,
TOT_ENR_M,97621,11,253,0.0,6890.0,"{'-5': 10, '-6': 1}",0,0,0,0.0
TOT_ENR_F,97621,11,798,0.0,7396.0,"{'-5': 10, '-6': 1}",0,0,0,0.0
SCH_ENR_IDEA_M,97622,10,3359,0.0,1243.0,{'-5': 10},0,0,0,0.0
SCH_ENR_IDEA_F,97622,10,4951,0.0,888.0,{'-5': 10},0,0,0,0.0
SCH_ENR_504_M,97622,10,27843,0.0,358.0,{'-5': 10},0,0,0,0.0
SCH_ENR_504_F,97622,10,33444,0.0,336.0,{'-5': 10},0,0,0,0.0
SCH_ENR_LEP_M,97622,10,23245,0.0,857.0,{'-5': 10},0,0,0,0.0
SCH_ENR_LEP_F,97622,10,25460,0.0,895.0,{'-5': 10},0,0,0,0.0
SCH_RSINSTANCES_SECL_WODIS,91141,6491,89079,0.0,474.0,"{'-6': 3089, '-9': 1908, '-11': 1203, '-5': 291}",0,0,0,0.0


In [15]:
summary = pd.DataFrame(NUMERIC_SUMMARY).set_index("source_column")
for cleaned in [e_clean, r_clean, s_clean, l_clean]:
    for col in cleaned:
        values = cleaned[col].dropna().astype(float)
        assert (values >= 0).all() and np.isfinite(values).all(), f"{col}: negative or infinite value left"
problem_cols = ["blank_cell", "unparseable_value", "non_finite_value", "non_integer_count"]
assert summary[problem_cols].fillna(0).to_numpy().sum() == 0, "blank/unparseable/non-finite/fraction found; see table"

n_reserved = sum(len(p) for p in quality_parts if p["issue"].iat[0] == "reserved_code")
print("No negatives or infinite values remain; 0 blank, unparseable, non-finite or fractional counts.")
compare("reserved-code values logged (all 21 count/FTE fields)", n_reserved, 22_835)
compare("valid seclusion WODIS values", int(r_clean["seclusion_instances_wodis"].notna().sum()), 91_141)
compare("valid TOT_ENR_M values", int(e_clean["tot_enr_m"].notna().sum()), 97_621)

No negatives or infinite values remain; 0 blank, unparseable, non-finite or fractional counts.
  reserved-code values logged (all 21 count/FTE fields) expected   22,835   actual   22,835   match
  valid seclusion WODIS values                         expected   91,141   actual   91,141   match
  valid TOT_ENR_M values                               expected   97,621   actual   97,621   match


**New York City check** (LEAID 3620580). Phase 2 found `SCH_FTESECURITY_LEO = -6.00` at all 1,603
NYC schools and `SCH_FTESECURITY_GUA = -6.00` at 1,595. The table below shows what the other 8 GUA
values are. TD section 7.1 p.11 says NYC's -6 codes should have been action-plan codes; they are logged as
they appear, with that note in the evidence.

In [16]:
nyc = mods["S"]["leaid"] == NYC_LEAID
nyc_view = mods["S"].loc[nyc, ["SCH_FTESECURITY_LEO", "SCH_FTESECURITY_GUA"]].assign(
    JJ=mods["S"].loc[nyc, "combokey"].map(dict(zip(c["combokey"], c["JJ"]))))
print(f"NYC schools: {nyc.sum():,}")
print(nyc_view.value_counts().rename("schools").to_string())

NYC schools: 1,603
SCH_FTESECURITY_LEO  SCH_FTESECURITY_GUA  JJ 
-6.00                -6.00                No     1595
                     -9.00                Yes       8


## Step 4 - Categories (playbook section 1, step 4)
`JJ`, the four school-status fields and the grade flags become exactly `Yes` / `No`, ignoring case
and spacing. Anything else (including a blank) becomes NA and is logged; a missing value never
becomes `No`. 2017-18 uses numeric category codes nowhere in these fields. `jj` comes from `JJ`
(`SCH_JUST_IND` does not exist this year).

In [17]:
def clean_yes_no(frame, raw_col, *, source_file="School Characteristics.csv"):
    # 'Yes'/'No' (case- and space-insensitive) -> 'Yes'/'No'; everything else NA and logged.
    norm = frame[raw_col].str.replace(r"\s+", "", regex=True).str.lower()
    out = pd.Series(pd.NA, index=frame.index, dtype="string")
    out[norm == "yes"] = "Yes"
    out[norm == "no"] = "No"
    common = dict(source_file=source_file, raw_column=raw_col, raw_value=frame[raw_col], action="set_na")
    log_issues(frame, out.isna() & frame[raw_col].eq(""), issue="blank_cell", **common)
    log_issues(frame, out.isna() & frame[raw_col].ne(""), issue="unexpected_category",
               evidence="Only Yes/No are defined (2017-18 CRDC File Structure.xlsx)", **common)
    return out


CATEGORY_COLS = ["JJ", *STATUS_COLS, *GRADE_COLS]
cat = pd.DataFrame({col: clean_yes_no(c, col) for col in CATEGORY_COLS})
cat_counts = pd.DataFrame({col: cat[col].value_counts(dropna=False) for col in CATEGORY_COLS}).T.fillna(0).astype(int)
cat_counts

,No,Yes
JJ,97030,602
SCH_STATUS_SPED,95421,2211
SCH_STATUS_CHARTER,90583,7049
SCH_STATUS_ALT,94289,3343
SCH_STATUS_MAGNET,93509,4123
SCH_GRADE_KG,45528,52104
SCH_GRADE_G01,44994,52638
SCH_GRADE_G02,44879,52753
SCH_GRADE_G03,44919,52713
SCH_GRADE_G04,45242,52390


In [18]:
assert cat.notna().all().all(), "Unexpected or blank category values; see quality log"
assert vc(cat["JJ"]) == {"No": 97_030, "Yes": 602}, "JJ counts differ from Phase 1"
print("All category values are Yes/No; nothing set to NA. JJ: 602 Yes / 97,030 No.")
print("Not cleaned (no output column), raw values for reference:",
      {col: vc(c[col]) for col in ["SCH_GRADE_PS", "SCH_GRADE_UG"]})

All category values are Yes/No; nothing set to NA. JJ: 602 Yes / 97,030 No.
Not cleaned (no output column), raw values for reference: {'SCH_GRADE_PS': {'No': 67111, 'Yes': 30520, '-6': 1}, 'SCH_GRADE_UG': {'No': 92999, 'Yes': 4632, '-6': 1}}


## Step 5 - Join the school modules (playbook section 1, step 5, section 5 Person B)
School Characteristics is the roster. Enrollment, Restraint and Seclusion and School Support are
left-joined on `COMBOKEY` with `validate="one_to_one"`, after selecting only the needed columns.
Restraint and Seclusion's `SCHID` is dropped **before** its join; `schid` always comes from the
roster. Key sets are compared before every join, and the roster row count must never change.

In [19]:
school = pd.DataFrame({
    "combokey": c["combokey"], "leaid": c["leaid"], "schid": c["schid"],
    "lea_state": c["LEA_STATE"].str.upper(), "sch_name": c["SCH_NAME"],
    "lea_name_c": c["LEA_NAME"], "_row_C": c["_row"],
    "ps_raw": c["SCH_GRADE_PS"], "ug_raw": c["SCH_GRADE_UG"],   # only for the grade_band breakdown
})
school = pd.concat([school, cat], axis=1)

r_raw_text = mods["R"][["combokey", "SCHID", *SECL_COLS]].copy()   # kept only for the join evidence below
join_frames = {
    "E": pd.concat([mods["E"][["combokey", "_row"]], e_clean], axis=1).rename(columns={"_row": "_row_E"}),
    "R": pd.concat([mods["R"][["combokey", "_row"]], r_clean], axis=1).rename(columns={"_row": "_row_R"}),
    "S": pd.concat([mods["S"][["combokey", "_row"]], s_clean], axis=1).rename(columns={"_row": "_row_S"}),
}
assert "SCHID" not in join_frames["R"].columns, "Restraint and Seclusion SCHID must be dropped before joining"

In [20]:
def compare_keys(left, right, key, label):
    # Print shared / missing / extra key counts before a join.
    left_keys, right_keys = set(left[key]), set(right[key])
    missing, extra = sorted(left_keys - right_keys), sorted(right_keys - left_keys)
    print(f"{label}: shared {len(left_keys & right_keys):,} | roster keys missing from module {len(missing):,}"
          f" {missing[:5]} | module keys not in roster {len(extra):,} {extra[:5]}")
    return missing, extra


JOIN_LOG = {}
missing_any = pd.Series(False, index=school.index)
for key, frame in join_frames.items():
    missing, extra = compare_keys(school, frame, "combokey", INPUTS[key].name)
    before = len(school)
    school = school.merge(frame, on="combokey", how="left", validate="one_to_one", indicator=True)
    no_row = school["_merge"] == "left_only"
    log_issues(school, no_row, source_file="School Characteristics.csv", row_col="_row_C", raw_column="COMBOKEY",
               raw_value=school["combokey"], issue="missing_module_row", action="fields_set_na",
               evidence=f"No row in {INPUTS[key].name}")
    missing_any = missing_any | no_row.to_numpy()
    JOIN_LOG[key] = {"rows_before": before, "rows_after": len(school), "matched": int((~no_row).sum()),
                     "missing": len(missing), "extra": len(extra)}
    school = school.drop(columns="_merge")
    assert len(school) == before == N_ROSTER, f"Row count changed in the {key} join"
school["flag_missing_module"] = missing_any.to_numpy()
pd.DataFrame(JOIN_LOG).T

Enrollment.csv: shared 97,632 | roster keys missing from module 0 [] | module keys not in roster 0 []


Restraint and Seclusion.csv: shared 97,632 | roster keys missing from module 0 [] | module keys not in roster 0 []


School Support.csv: shared 97,632 | roster keys missing from module 0 [] | module keys not in roster 0 []


,rows_before,rows_after,matched,missing,extra
E,97632,97632,97632,0,0
R,97632,97632,97632,0,0
S,97632,97632,97632,0,0


In [21]:
assert (school["combokey"].to_numpy() == c["combokey"].to_numpy()).all(), "Roster order changed during joins"
assert not school["combokey"].duplicated().any()
assert not school["flag_missing_module"].any(), "Some roster schools have no row in a module"
STEP_ROWS.append(("5 school modules joined", len(school)))
print(f"All three joins matched {N_ROSTER:,} of {N_ROSTER:,} roster schools; 0 extra keys; row order unchanged.")

All three joins matched 97,632 of 97,632 roster schools; 0 extra keys; row order unchanged.


**Restraint and Seclusion join evidence** (the team will spot-check this join). Two checks:
(1) for every school, the cleaned seclusion values equal the raw text in Restraint and Seclusion
looked up by `COMBOKEY`; (2) three sample schools (first, middle and last roster schools with all
three components > 0), showing that R's own `SCHID` disagrees with the roster but was not used.

In [22]:
lookup = school[["combokey", "schid", "sch_name", *SECL_COLS.values()]].merge(
    r_raw_text, on="combokey", how="left", validate="one_to_one")
for raw_col, out_col in SECL_COLS.items():
    raw_num = pd.to_numeric(lookup[raw_col])
    expected = raw_num.where(raw_num >= 0)
    same = (lookup[out_col].astype(float).fillna(-1) == expected.fillna(-1)).all()
    assert same, f"{out_col} does not equal the raw Restraint and Seclusion value on every row"
print("Check 1: all 3 seclusion columns equal the raw R values on all", f"{len(lookup):,}", "rows.")

all_positive = lookup[list(SECL_COLS.values())].gt(0).fillna(False).all(axis=1)
pos_idx = lookup.index[all_positive]
SAMPLE_R = lookup.loc[[pos_idx[0], pos_idx[len(pos_idx) // 2], pos_idx[-1]]]
print(f"Schools with all three components > 0: {len(pos_idx):,}. Samples:")
SAMPLE_R.rename(columns={"schid": "schid (roster)", "SCHID": "SCHID (R, ignored)"})

Check 1: all 3 seclusion columns equal the raw R values on all 97,632 rows.
Schools with all three components > 0: 146. Samples:


,combokey,schid (roster),sch_name,seclusion_instances_wodis,seclusion_instances_idea,seclusion_instances_504,"SCHID (R, ignored)",SCH_RSINSTANCES_SECL_WODIS,SCH_RSINSTANCES_SECL_IDEA,SCH_RSINSTANCES_SECL_504
1022,010243001022,01022,Highland Gardens Elementary School,3.0,4.0,1.0,01010,3,4,1
39523,240066001667,01667,Rockland Woods Elementary,24.0,195.0,23.0,24016,24,195,23
95854,551464002756,02756,Creekside Elementary,3.0,4.0,2.0,55027,3,4,2


## Step 6 - Join the district file (playbook section 1, step 6)
LEA Characteristics is left-joined on `LEAID` with `validate="many_to_one"`. `lea_name`,
`lea_enrollment` and `lea_schools` come from the LEA file; school IDs and names stay from the roster.
The roster's `LEA_NAME` / `LEA_STATE` are compared with the LEA file and any disagreement is logged.
Names from Enrollment and Restraint and Seclusion are not used (R cuts `LEA_NAME` at 50 characters;
E has a few double-encoded characters); those are QA notes, not per-row issues.

In [23]:
lea_frame = pd.concat([lea[["leaid", "LEA_NAME", "LEA_STATE", "_row"]], l_clean], axis=1).rename(
    columns={"LEA_NAME": "lea_name", "LEA_STATE": "lea_state_l", "_row": "_row_L"})
missing, extra = compare_keys(school, lea_frame, "leaid", "LEA Characteristics.csv (by LEAID)")
before = len(school)
school = school.merge(lea_frame, on="leaid", how="left", validate="many_to_one", indicator=True)
school["flag_missing_lea"] = school["_merge"] == "left_only"
log_issues(school, school["flag_missing_lea"], source_file="School Characteristics.csv", row_col="_row_C",
           raw_column="LEAID", raw_value=school["leaid"], issue="missing_lea_row", action="fields_set_na",
           evidence="No row in LEA Characteristics.csv")
school = school.drop(columns="_merge")
assert len(school) == before == N_ROSTER, "Row count changed in the LEA join"
assert (school["combokey"].to_numpy() == c["combokey"].to_numpy()).all(), "Roster order changed in the LEA join"
print(f"Matched {int((~school['flag_missing_lea']).sum()):,} of {N_ROSTER:,} schools; "
      f"LEAs with no roster school: {len(extra):,}")

LEA Characteristics.csv (by LEAID): shared 17,604 | roster keys missing from module 0 [] | module keys not in roster 0 []
Matched 97,632 of 97,632 schools; LEAs with no roster school: 0


In [24]:
for c_col, l_col, frame_col in [("lea_name_c", "lea_name", "LEA_NAME"), ("lea_state", "lea_state_l", "LEA_STATE")]:
    differs = school[l_col].notna() & (school[c_col] != school[l_col])
    n = log_issues(school, differs, source_file="School Characteristics.csv", row_col="_row_C",
                   raw_column=frame_col, raw_value=school[c_col], issue="name_disagreement",
                   action="roster value kept for school IDs/names; LEA file value used for lea_name",
                   evidence="LEA Characteristics.csv value: " + school[l_col].astype(str))
    compare(f"roster vs LEA file {frame_col} disagreements", n, 0)

assert not school["flag_missing_lea"].any(), "Some schools have no LEA row"
assert school["lea_state"].str.fullmatch(r"[A-Z]{2}").all(), "lea_state is not 2 uppercase letters"
compare("states/territories (incl. PR)", school["lea_state"].nunique(), 52)
STEP_ROWS.append(("6 LEA file joined", len(school)))

  roster vs LEA file LEA_NAME disagreements            expected        0   actual        0   match
  roster vs LEA file LEA_STATE disagreements           expected        0   actual        0   match
  states/territories (incl. PR)                        expected       52   actual       52   match


## Step 7 - Seclusion total (playbook section 1, step 7, section 4)
`seclusion_instances` = WODIS + IDEA + 504, only when all three are valid; otherwise NA (the strict
rule, Phase 1 decision 2). Phase 2 verified the three groups are mutually exclusive and are counts of
incidents, not students (ML "Seclusion" PDF pp.22-23; MAN sec. 3.4 p.9; `mapping_201718.csv` lines 33-35).
Student counts (`TOT_RS_*`) are never read.

In [25]:
SECLUSION_GROUPS_VERIFIED = True   # Phase 2 / Checkpoint 2; set False to force the total and rate to NA
SECL_OUT = list(SECL_COLS.values())


def seclusion_total(frame):
    parts = frame[SECL_OUT]
    all_valid = parts.notna().all(axis=1)
    total = parts.sum(axis=1, min_count=len(SECL_OUT)).where(all_valid).astype("Float64")
    return total if SECLUSION_GROUPS_VERIFIED else pd.Series(pd.NA, index=frame.index, dtype="Float64")


school["seclusion_instances"] = seclusion_total(school)

In [26]:
valid_total = school["seclusion_instances"].notna()
assert (valid_total == school[SECL_OUT].notna().all(axis=1)).all(), "Total is valid where a component is NA"
assert (school.loc[valid_total, "seclusion_instances"]
        == school.loc[valid_total, SECL_OUT].sum(axis=1)).all(), "Total != sum of components"
compare("schools with a valid seclusion total", int(valid_total.sum()), 91_013)
compare("schools with a NA seclusion total", int((~valid_total).sum()), 6_619)
compare("valid total > 0", int((school["seclusion_instances"] > 0).sum()), 5_975)
compare("New York City schools: valid total, all 0",
        int((school.loc[school["leaid"] == NYC_LEAID, "seclusion_instances"] == 0).sum()), 1_603)
STEP_ROWS.append(("7 seclusion total", len(school)))

  schools with a valid seclusion total                 expected   91,013   actual   91,013   match
  schools with a NA seclusion total                    expected    6,619   actual    6,619   match
  valid total > 0                                      expected    5,975   actual    5,975   match
  New York City schools: valid total, all 0            expected    1,603   actual    1,603   match


## Step 8 - Enrollment totals (playbook section 1, step 8, section 4)
Total, IDEA, 504 and EL enrollment are each M + F, and NA unless **both** M and F are valid (pandas'
`.sum()` would silently skip NA, so the rule is written out). 2017-18 has no nonbinary (X) counts, so
`enr_nonbinary` is NA and every `flag_partial_*` is False. EL comes from `SCH_ENR_LEP_M/F` (LEP = EL,
ML pp.9-10). Caveat (Phase 2 decision 1): `TOT_ENR` includes preschool, while seclusion appears to
cover K-12/ungraded only. The playbook formula is kept pending a team decision.

In [27]:
def strict_mf_total(m, f):
    # M + F only when both are valid; NA otherwise.
    return (m + f).where(m.notna() & f.notna()).astype("Float64")


school["enrollment_total"] = strict_mf_total(school["tot_enr_m"], school["tot_enr_f"])
school["enr_idea"] = strict_mf_total(school["idea_m"], school["idea_f"])
school["enr_504"] = strict_mf_total(school["s504_m"], school["s504_f"])
school["enr_el"] = strict_mf_total(school["el_m"], school["el_f"])
school["enr_nonbinary"] = pd.Series(pd.NA, index=school.index, dtype="Float64")   # not collected in 2017-18
for name in ["enrollment", "idea", "504", "el"]:
    school[f"flag_partial_{name}"] = False                                         # no X component this year

In [28]:
for total, (m, f) in {"enrollment_total": ("tot_enr_m", "tot_enr_f"), "enr_idea": ("idea_m", "idea_f"),
                      "enr_504": ("s504_m", "s504_f"), "enr_el": ("el_m", "el_f")}.items():
    one_missing = school[m].isna() | school[f].isna()
    assert school.loc[one_missing, total].isna().all(), f"{total} has a value where M or F is NA"
    assert school.loc[~one_missing, total].notna().all(), f"{total} is NA where M and F are both valid"
    print(f"  {total:<17} valid {int(school[total].notna().sum()):>7,}  NA {int(school[total].isna().sum()):>3,}")
compare("enrollment_total NA (M or F reserved)", int(school["enrollment_total"].isna().sum()), 11)
compare("enrollment_total == 0", int((school["enrollment_total"] == 0).sum()), 0)
STEP_ROWS.append(("8 enrollment totals", len(school)))

  enrollment_total  valid  97,621  NA  11
  enr_idea          valid  97,622  NA  10
  enr_504           valid  97,622  NA  10
  enr_el            valid  97,622  NA  10
  enrollment_total NA (M or F reserved)                expected       11   actual       11   match
  enrollment_total == 0                                expected        0   actual        0   match


## Step 9 - Derived features and flags (playbook section 4)
Rates are per 100 students: `100 * numerator / enrollment_total`, only with a valid numerator and a
positive denominator, so they are never infinite. Seclusion rates above 100 are allowed. Subgroup
percentages also need subgroup <= total; when a subgroup is larger than the total, the percentage is
NA, the components are kept, and the row is logged. `grade_band` follows the section 4 rule order.

In [29]:
def per_100(numerator, denominator):
    ok = numerator.notna() & denominator.notna() & (denominator > 0)
    return (100 * numerator / denominator).where(ok).astype("Float64")


def subgroup_pct(subgroup, total):
    ok = subgroup.notna() & total.notna() & (total > 0) & (subgroup <= total)
    return (100 * subgroup / total).where(ok).astype("Float64")


den = school["enrollment_total"]
school["seclusion_rate"] = per_100(school["seclusion_instances"], den)
school["teachers_per_100"] = per_100(school["fte_teachers"], den)
school["counselors_per_100"] = per_100(school["fte_counselors"], den)
def support_per_100(frame, denominator):
    # Nurses + psychologists + social workers per 100 students; NA unless all three are valid.
    parts = frame[["fte_nurses", "fte_psych", "fte_social"]]
    total = parts.sum(axis=1, min_count=3).where(parts.notna().all(axis=1)).astype("Float64")
    return per_100(total, denominator)


school["support_fte_per_100"] = support_per_100(school, den)

SUBGROUPS = {"pct_idea": ("enr_idea", "SCH_ENR_IDEA_M;SCH_ENR_IDEA_F"),
             "pct_504": ("enr_504", "SCH_ENR_504_M;SCH_ENR_504_F"),
             "pct_el": ("enr_el", "SCH_ENR_LEP_M;SCH_ENR_LEP_F")}
for pct_col, (sub_col, raw_cols) in SUBGROUPS.items():
    school[pct_col] = subgroup_pct(school[sub_col], den)
    above = school[sub_col].notna() & den.notna() & (school[sub_col] > den)
    n = log_issues(school, above, source_file="Enrollment.csv", row_col="_row_E", raw_column=raw_cols,
                   raw_value=school[sub_col].astype(str), issue="subgroup_above_total",
                   action=f"{pct_col} set to NA; components retained",
                   evidence=f"{sub_col} (M+F) > enrollment_total (TOT_ENR_M+TOT_ENR_F) = " + den.astype(str))
    print(f"  {pct_col}: subgroup > total on {n} rows")

  pct_idea: subgroup > total on 222 rows


  pct_504: subgroup > total on 0 rows
  pct_el: subgroup > total on 10 rows


In [30]:
GRADE_E = ["SCH_GRADE_KG"] + [f"SCH_GRADE_G{g:02d}" for g in range(1, 6)]
GRADE_M = [f"SCH_GRADE_G{g:02d}" for g in range(6, 9)]
GRADE_H = [f"SCH_GRADE_G{g:02d}" for g in range(9, 13)]


def grade_band(flags):
    # Playbook sec. 4: NA if any KG/G01-G12 flag is unknown; 2-3 of E/M/H -> span; one -> that band; none -> other.
    yes = flags[GRADE_COLS].eq("Yes")
    e, m, h = yes[GRADE_E].any(axis=1), yes[GRADE_M].any(axis=1), yes[GRADE_H].any(axis=1)
    n_bands = e.astype(int) + m.astype(int) + h.astype(int)
    band = pd.Series(np.select([n_bands >= 2, e, m, h], ["span", "elementary", "middle", "high"], "other"),
                     index=flags.index, dtype="string")
    return band.mask(flags[GRADE_COLS].isna().any(axis=1))


school["grade_band"] = grade_band(school)

In [31]:
school["year"] = YEAR
school["jj"] = school["JJ"]
for raw_col, out_col in STATUS_COLS.items():
    school[out_col] = school[raw_col]
school["sch_virt_ind"] = pd.Series(pd.NA, index=school.index, dtype="string")   # not collected in 2017-18

def outcome_flags(seclusion_instances, enrollment_total):
    # Playbook sec. 4 flag definitions (flag_analysis_ready per Phase 2 decision 1).
    return pd.DataFrame({
        "flag_missing_seclusion": seclusion_instances.isna(),
        "flag_missing_enrollment": enrollment_total.isna() | (enrollment_total <= 0).fillna(False),
        "flag_small_enrollment": (enrollment_total < 10).fillna(False),
        "flag_analysis_ready": seclusion_instances.notna() & (enrollment_total > 0).fillna(False),
    }).astype(bool)


school = school.join(outcome_flags(school["seclusion_instances"], school["enrollment_total"]))

In [32]:
RATE_COLS = ["seclusion_rate", "teachers_per_100", "counselors_per_100", "support_fte_per_100",
             "pct_idea", "pct_504", "pct_el"]
for col in RATE_COLS:
    values = school[col].to_numpy(dtype=float, na_value=np.nan)
    assert np.isfinite(values[~school[col].isna().to_numpy()]).all(), f"{col}: NaN/inf hidden among non-NA values"
    assert (school[col].dropna() >= 0).all(), f"{col}: negative value"
    assert school.loc[den.isna() | (den <= 0).fillna(False), col].isna().all(), f"{col}: value without a positive denominator"
for col in ["pct_idea", "pct_504", "pct_el"]:
    assert (school[col].dropna() <= 100).all(), f"{col}: percentage above 100"
assert school["grade_band"].notna().all(), "grade_band is NA for some schools"

print("Rate summary (valid count, max):")
for col in RATE_COLS:
    print(f"  {col:<20} valid {int(school[col].notna().sum()):>7,}  max {school[col].max():>12,.2f}")
print("seclusion_rate > 100:", int((school["seclusion_rate"] > 100).sum()), "schools (allowed)")

Rate summary (valid count, max):
  seclusion_rate       valid  91,013  max     3,250.00
  teachers_per_100     valid  97,611  max     2,072.00
  counselors_per_100   valid  97,611  max     1,500.00
  support_fte_per_100  valid  97,611  max     2,138.05
  pct_idea             valid  97,399  max       100.00
  pct_504              valid  97,621  max       100.00
  pct_el               valid  97,611  max       100.00
seclusion_rate > 100: 155 schools (allowed)


In [33]:
print("grade_band:", vc(school["grade_band"]))
other = school["grade_band"] == "other"
k12_none = ~school[GRADE_COLS].eq("Yes").any(axis=1)
ps, ug = school["ps_raw"].eq("Yes"), school["ug_raw"].eq("Yes")
print("'other' breakdown: preschool only", int((other & ps & ~ug).sum()), "| ungraded only", int((other & ug & ~ps).sum()),
      "| preschool + ungraded", int((other & ps & ug).sum()), "| no grade flag at all", int((other & ~ps & ~ug).sum()))
assert (other == k12_none).all(), "'other' should be exactly the schools with no KG-G12 grade"
STEP_ROWS.append(("9 derived features", len(school)))

grade_band: {'elementary': 36302, 'span': 28842, 'high': 17234, 'middle': 13070, 'other': 2184}
'other' breakdown: preschool only 1740 | ungraded only 224 | preschool + ungraded 39 | no grade flag at all 181


## Step 10 - FTE review (playbook section 1, step 10, section 4)
Review triggers: teachers > 2,000 or any other core FTE > 200 (Phase 1 decision 3). In addition, the
top 10 values of each FTE field were reviewed at Checkpoint 3 against enrollment, teacher FTE, the same
field at the LEA's other schools ("peer max") and OCR's post-submission checks (Appendix C, STAF sheet).
Approved values are flagged as "identified as suspicious during review" (playbook section 4), by group:
**A** = at least 15x the peer max, or peers all 0 (consistent with a x100 / decimal entry; cause not
proven); **B** = equal to the school's teacher FTE; **C** = out of line with enrollment or teacher FTE
without a peer pattern; **LEA cluster** = every school in the LEA reporting the same value in that field.
Flagged values are **kept as reported**; each gets one `fte_outlier` row with its evidence. Justice
facilities and values with plausible ratios were not flagged. This review is not exhaustive.

In [34]:
FTE_TRIGGER = {col: (2000 if col == "fte_teachers" else 200) for col in FTE_COLS.values()}
FTE_RAW = {new: raw_col for raw_col, new in FTE_COLS.items()}

# OCR post-submission DQ flags per LEA (Appndx C. Post-Sub Data Quality Checks.xlsx, STAF sheet),
# for the LEAs below; looked up at Phase 2 (D8) and Checkpoint 3. Check 84 = counselors, nurses,
# psychologists or security staff >= teachers (not applicable to justice facilities).
OCR_STAF = {
    "0403780": "not listed", "0603840": "Check 84", "0606360": "Check 84", "0630780": "Check 84",
    "0633750": "Check 84", "0904830": "Check 84", "0904980": "not listed", "1601590": "Check 84",
    "1718840": "not listed", "1727390": "Check 11, Check 84", "1738220": "not listed", "1741040": "Check 84",
    "1803750": "Check 84", "1804770": "Check 84, Multivariate STAF", "1810650": "Check 84, Univariate STAF",
    "1811340": "not listed", "1812810": "Univariate STAF", "2011490": "not listed", "2012420": "Check 84",
    "2105910": "Check 84", "2200047": "Check 84", "2400270": "Check 84", "2503330": "not listed",
    "2700099": "not listed", "2801620": "Check 84", "3175770": "not listed", "3200001": "Check 11",
    "3304590": "Check 84", "3500148": "Check 84", "3500153": "Check 84", "3604920": "not listed",
    "4000800": "not listed", "4200806": "Check 84", "4217520": "not listed", "4816620": "Check 84, Univariate STAF",
    "4835570": "not listed", "5103750": "not listed", "5305910": "Check 15, Multivariate STAF", "5517040": "Check 84",
}

# Approved at Checkpoint 3 (decision 1): (combokey, field, group).
APPROVED_SUSPICIOUS_FTE = [
    ("400080002865", "fte_teachers", "C"),
    ("280162000249", "fte_counselors", "A"), ("350015301083", "fte_counselors", "C"),
    ("350014801089", "fte_counselors", "B"), ("174104004109", "fte_counselors", "A"),
    ("250333000457", "fte_counselors", "C"), ("210591001939", "fte_counselors", "B"),
    ("201242000068", "fte_counselors", "A"), ("201242000070", "fte_counselors", "A"),
    ("172739002908", "fte_counselors", "C"),
    ("181134001821", "fte_nurses", "A"), ("510375001201", "fte_nurses", "A"),
    ("181281000238", "fte_nurses", "A+B"), ("201242000067", "fte_nurses", "A"),
    ("421752005272", "fte_nurses", "A"), ("173822004522", "fte_nurses", "A"),
    ("060636000560", "fte_psych", "C"), ("181065001750", "fte_psych", "A"),
    ("181281000238", "fte_psych", "A+B"), ("090483001638", "fte_psych", "C"),
    ("201149000693", "fte_psych", "A"), ("040378002477", "fte_psych", "A"),
    ("510375002002", "fte_psych", "A"),
    ("240027001727", "fte_social", "A"), ("090483001638", "fte_social", "A"),
    ("270009904609", "fte_social", "A"), ("171884002170", "fte_social", "A"),
    ("320000100908", "fte_social", "A"), ("330459000257", "fte_social", "A"),
    ("360492000236", "fte_social", "A"), ("510375002002", "fte_social", "A"),
    ("551704002200", "fte_leo", "A"), ("180375002436", "fte_leo", "A"),
    ("483557007057", "fte_leo", "A"), ("063078004785", "fte_leo", "A"),
    ("180477000878", "fte_guards", "A"), ("060384013223", "fte_guards", "A"),
]
# LEA clusters (decision 1): flag every school in the LEA with exactly this value in this field.
APPROVED_FTE_CLUSTERS = [("0633750", "fte_nurses", 33.33), ("3175770", "fte_psych", 33.30),
                         ("0904980", "fte_social", 33.00)]
GROUP_REASON = {
    "A": "value >= 15x the largest value at the LEA's other schools, or those are all 0 (consistent with a x100 or decimal entry, cause not proven)",
    "B": "value equals the school's teacher FTE",
    "A+B": "value >= 15x the largest value at the LEA's other schools and equal to the school's teacher FTE",
    "C": "out of line with enrollment or teacher FTE, no peer pattern",
}
SUSPICIOUS_NOTE = "identified as suspicious during FTE review (playbook sec. 4), approved at Checkpoint 3"
def fmt(x):
    return "NA" if pd.isna(x) else f"{float(x):.10g}"


def fte_evidence(row, col, group, reason):
    # Evidence text for one flagged FTE value: group, value, enrollment, teacher FTE, peer max, OCR flags.
    peers = school.loc[(school["leaid"] == row["leaid"]) & (school["combokey"] != row["combokey"]), col]
    peer = "single-school LEA" if peers.empty else f"{fmt(peers.max())} (other schools in LEA: {len(peers)})"
    return (f"Group {group}: {reason}; value {fmt(row[col])}; enrollment_total {fmt(row['enrollment_total'])}; "
            f"fte_teachers {fmt(row['fte_teachers'])}; peer max {peer}; "
            f"OCR Appndx C STAF: {OCR_STAF.get(row['leaid'], 'not listed')}; value kept as reported")


s_text = mods["S"].set_index("_row")
FTE_LOGGED = []   # (combokey, field, group) of every fte_outlier row


def flag_fte(hit, col, group, reason):
    # Set flag_fte_outlier on `hit` rows and log one fte_outlier row each (value kept).
    hit = hit.fillna(False).astype(bool)
    school.loc[hit, "flag_fte_outlier"] = True
    evidence = pd.Series("", index=school.index)
    for i in school.index[hit]:
        evidence[i] = fte_evidence(school.loc[i], col, group, reason)
        FTE_LOGGED.append((school.at[i, "combokey"], col, group))
    return log_issues(school, hit, source_file="School Support.csv", row_col="_row_S", raw_column=FTE_RAW[col],
                      raw_value=school["_row_S"].map(s_text[FTE_RAW[col]]), issue="fte_outlier",
                      action="retained; flag_fte_outlier = True", cleaned_value=school[col].astype(str),
                      evidence=evidence)


school["flag_fte_outlier"] = False
for col, cutoff in FTE_TRIGGER.items():
    flag_fte(school[col] > cutoff, col, "trigger", f"review trigger {col} > {cutoff:,} (playbook sec. 1 step 10)")
for combokey, col, group in APPROVED_SUSPICIOUS_FTE:
    hit = school["combokey"] == combokey
    assert hit.sum() == 1, f"{combokey} not found"
    flag_fte(hit, col, group, f"{GROUP_REASON[group]}; {SUSPICIOUS_NOTE}")
for leaid, col, value in APPROVED_FTE_CLUSTERS:
    hit = (school["leaid"] == leaid) & (school[col] == value)
    flag_fte(hit, col, "LEA cluster",
             f"all {int(hit.sum())} schools in LEA {leaid} report {col} = {value:g}; {SUSPICIOUS_NOTE}")

fte_log = pd.DataFrame(FTE_LOGGED, columns=["combokey", "field", "group"])
fte_log.groupby("group").agg(rows=("combokey", "size"), schools=("combokey", "nunique"))

,rows,schools
group,,
A,27,26
A+B,2,1
B,2,2
C,6,6
LEA cluster,24,24
trigger,10,10


In [35]:
assert not fte_log.duplicated(["combokey", "field"]).any(), "An FTE value was logged twice"
compare("fte_outlier rows: review triggers", int((fte_log["group"] == "trigger").sum()), 10)
compare("fte_outlier rows: approved suspicious pairs", int(fte_log["group"].isin(GROUP_REASON).sum()), 37)
for leaid, col, value in APPROVED_FTE_CLUSTERS:
    n_cluster = int(((school["leaid"] == leaid) & (school[col] == value)).sum())
    compare(f"LEA cluster {leaid} {col} = {value:g} (schools)", n_cluster, 20 if leaid == "0633750" else 2)
compare("fte_outlier rows in total", len(fte_log), 71)
compare("schools with flag_fte_outlier", int(school["flag_fte_outlier"].sum()), 68)
assert fte_log["combokey"].nunique() == int(school["flag_fte_outlier"].sum())
assert school.loc[school["combokey"] == "481662001424", "fte_guards"].iat[0] == 102_579, "Del Valle value changed"
STEP_ROWS.append(("10 FTE review", len(school)))

  fte_outlier rows: review triggers                    expected       10   actual       10   match
  fte_outlier rows: approved suspicious pairs          expected       37   actual       37   match
  LEA cluster 0633750 fte_nurses = 33.33 (schools)     expected       20   actual       20   match
  LEA cluster 3175770 fte_psych = 33.3 (schools)       expected        2   actual        2   match
  LEA cluster 0904980 fte_social = 33 (schools)        expected        2   actual        2   match
  fte_outlier rows in total                            expected       71   actual       71   match
  schools with flag_fte_outlier                        expected       68   actual       68   match


Top 10 values per FTE field (the list reviewed at Checkpoint 3). `per_100` is the FTE per 100 enrolled
students; `teachers` is the school's teacher FTE; `flagged` shows `flag_fte_outlier` after the step above.

In [36]:
def top10(col):
    top = school.loc[school[col].notna()].sort_values([col, "combokey"], ascending=[False, True]).head(10)
    return pd.DataFrame({"field": col, "value": top[col].astype(float), "combokey": top["combokey"],
                         "sch_name": top["sch_name"].str[:32], "state": top["lea_state"],
                         "enrollment": top["enrollment_total"], "teachers": top["fte_teachers"],
                         "per_100": (100 * top[col] / top["enrollment_total"]).astype(float).round(1),
                         "jj": top["jj"], "flagged": top["flag_fte_outlier"]})


TOP10 = pd.concat([top10(col) for col in FTE_COLS.values()], ignore_index=True)
with pd.option_context("display.max_rows", 100):
    display(TOP10)

,field,value,combokey,sch_name,state,enrollment,teachers,per_100,jj,flagged
0,fte_teachers,607.00,340002400289,North Star Academy Charter Schoo,NJ,5462.0,607.0,11.1,No,False
1,fte_teachers,499.00,400080002865,EPIC BLENDED CHARTER TULSA MS,OK,485.0,499.0,102.9,No,True
2,fte_teachers,457.00,420014000896,Agora Cyber CS,PA,6022.0,457.0,7.6,No,False
3,fte_teachers,406.00,360012606464,NYS DEPT OF CORRECTIONS PLACEMEN,NY,887.0,406.0,45.8,No,False
4,fte_teachers,396.00,510012000054,T.C. Williams High,VA,3974.0,396.0,10.0,No,False
5,fte_teachers,387.00,130023204148,Georgia Cyber Academy,GA,14286.0,387.0,2.7,No,False
6,fte_teachers,362.00,420007100534,Pennsylvania Cyber CS,PA,9676.0,362.0,3.7,No,False
7,fte_teachers,361.00,341263004860,Passaic County Technical Institu,NJ,3447.0,361.0,10.5,No,False
8,fte_teachers,353.61,390018004704,Ohio Virtual Academy,OH,9287.0,353.61,3.8,No,False
9,fte_teachers,348.70,420011900756,Commonwealth Charter Academy CS,PA,8592.0,348.7,4.1,No,False


## Step 11 - Output (playbook section 3)
The output has exactly the 50 columns of playbook section 3.1, in order, with the section 3.2 types: IDs as
zero-padded text, numbers as nullable floats, flags as True/False, categories as `Yes`/`No`/NA.
CSV: UTF-8, header, no index, empty cell for NA, `\n` line endings. Rows stay in roster order, so a
rerun gives byte-identical files.

In [37]:
OUTPUT_COLUMNS = [
    "year", "combokey", "leaid", "schid", "lea_state", "sch_name", "lea_name", "jj",
    "seclusion_instances_wodis", "seclusion_instances_idea", "seclusion_instances_504", "seclusion_instances",
    "enrollment_total", "enr_nonbinary", "enr_idea", "enr_504", "enr_el", "seclusion_rate",
    "sch_status_sped", "sch_status_charter", "sch_status_alt", "sch_status_magnet", "sch_virt_ind", "grade_band",
    "fte_teachers", "fte_counselors", "fte_nurses", "fte_psych", "fte_social", "fte_leo", "fte_guards",
    "counselors_per_100", "teachers_per_100", "support_fte_per_100", "pct_idea", "pct_504", "pct_el",
    "lea_enrollment", "lea_schools",
    "flag_missing_seclusion", "flag_missing_enrollment", "flag_small_enrollment", "flag_fte_outlier",
    "flag_partial_enrollment", "flag_partial_idea", "flag_partial_504", "flag_partial_el",
    "flag_missing_module", "flag_missing_lea", "flag_analysis_ready",
]
TEXT_COLS = ["year", "combokey", "leaid", "schid", "lea_state", "sch_name", "lea_name"]
CATEGORY_OUT = ["jj", *STATUS_COLS.values(), "sch_virt_ind", "grade_band"]
FLAG_COLS = [col for col in OUTPUT_COLUMNS if col.startswith("flag_")]
NUMBER_COLS = [col for col in OUTPUT_COLUMNS if col not in TEXT_COLS + CATEGORY_OUT + FLAG_COLS]

full = school[OUTPUT_COLUMNS].copy()
full[TEXT_COLS] = full[TEXT_COLS].astype(str)
full[CATEGORY_OUT] = full[CATEGORY_OUT].astype("string")
full[NUMBER_COLS] = full[NUMBER_COLS].astype("Float64")
full[FLAG_COLS] = full[FLAG_COLS].astype(bool)
analysis = full.loc[full["flag_analysis_ready"]]

In [38]:
assert len(OUTPUT_COLUMNS) == 50 and len(set(OUTPUT_COLUMNS)) == 50, "Schema must have 50 unique columns"
assert list(full.columns) == OUTPUT_COLUMNS
assert len(full) == N_ROSTER and full["combokey"].is_unique
assert (full["year"] == YEAR).all()
assert full["combokey"].str.len().eq(12).all() and full["leaid"].str.len().eq(7).all() and full["schid"].str.len().eq(5).all()
assert full[TEXT_COLS].notna().all().all() and not full[TEXT_COLS].isin(["", "nan", "None"]).any().any()
for col in NUMBER_COLS:
    values = full[col].to_numpy(dtype=float, na_value=np.nan)
    assert np.isfinite(values[~full[col].isna().to_numpy()]).all(), f"{col}: NaN or inf"
    assert (full[col].dropna() >= 0).all(), f"{col}: negative value"
for col in CATEGORY_OUT[:-1]:
    assert (full[col].isna() | full[col].isin(["Yes", "No"])).all(), f"{col}: value other than Yes/No/NA"
assert (full["grade_band"].isna() | full["grade_band"].isin(["elementary", "middle", "high", "span", "other"])).all()
assert full["enr_nonbinary"].isna().all() and full["sch_virt_ind"].isna().all()
assert not full[["flag_partial_enrollment", "flag_partial_idea", "flag_partial_504", "flag_partial_el"]].any().any()
assert analysis.equals(full[full["flag_analysis_ready"]]) and analysis["seclusion_instances"].notna().all()
print(f"Schema OK: {full.shape[0]:,} rows x {full.shape[1]} columns; analysis subset {len(analysis):,} rows.")

Schema OK: 97,632 rows x 50 columns; analysis subset 91,013 rows.


**Quality-detail table.** Fields that do not exist in 2017-18 (`enr_nonbinary` from `TOT_ENR_X`,
`sch_virt_ind` from `SCH_VIRT_IND`) are **not** logged per school, which would add about 195,000
rows. Playbook section 3.3 allows a blank `combokey` only for district rows identified by `leaid`, so a row
with no IDs at all is not provided for. These fields are recorded in `mapping_201718.csv` (lines 53-58,
`not_collected`) and will be listed in the QA report instead.

In [39]:
quality = pd.concat(quality_parts, ignore_index=True)[QD_COLUMNS]
quality["source_row"] = quality["source_row"].astype("Int64")
QD_TEXT = [col for col in QD_COLUMNS if col != "source_row"]
quality[QD_TEXT] = quality[QD_TEXT].fillna("")
assert quality["source_row"].notna().all() and (quality["source_row"] >= 1).all()
print(f"quality_detail rows: {len(quality):,}; approx. CSV size {len(quality.to_csv(index=False)) / 1e6:.1f} MB")
print(quality["issue"].value_counts().to_string())

quality_detail rows: 23,648; approx. CSV size 5.2 MB
issue
reserved_code           22835
non_digit_id              510
subgroup_above_total      232
fte_outlier                71


In [40]:
CSV_OPTIONS = dict(index=False, encoding="utf-8", na_rep="", lineterminator="\n")
OUT_DIR.mkdir(parents=True, exist_ok=True)
full.to_csv(OUT_FULL, **CSV_OPTIONS)
analysis.to_csv(OUT_ANALYSIS, **CSV_OPTIONS)
quality.to_csv(OUT_QUALITY, **CSV_OPTIONS)

for path, frame in [(OUT_FULL, full), (OUT_ANALYSIS, analysis), (OUT_QUALITY, quality)]:
    print(f"{rel(path)}: {len(frame):,} rows x {frame.shape[1]} cols, {path.stat().st_size:,} bytes, sha256 {sha256(path)}")
STEP_ROWS.append(("11 output written", len(full)))

data/processed/201718/schools_201718.csv: 97,632 rows x 50 cols, 34,286,277 bytes, sha256 2a1fe1c873a8cc7fe60e7f79752c7d036e155095769cf2365d0391027556baaa
data/processed/201718/schools_201718_analysis.csv: 91,013 rows x 50 cols, 32,079,534 bytes, sha256 4c0892743767bbfdfa24f26cd41eb550284cf2d1fc32f96c25402e19f9f8c3f6
data/processed/201718/quality_detail_201718.csv: 23,648 rows x 11 cols, 5,178,633 bytes, sha256 6a3550d54c51a7864242add38e19f9a37d09ead964fc288044a919506f97784c


## Summary
Row counts after each step, NA counts per output column, flag counts next to the values expected
from Phases 1-2 ("compare, don't force"), and quality-detail counts by issue.

In [41]:
print("Roster rows after each step:")
for step, n in STEP_ROWS:
    print(f"  {step:<42} {n:>7,}")

Roster rows after each step:
  1 read (School Characteristics roster)      97,632
  2 IDs validated                             97,632
  5 school modules joined                     97,632
  6 LEA file joined                           97,632
  7 seclusion total                           97,632
  8 enrollment totals                         97,632
  9 derived features                          97,632
  10 FTE review                               97,632
  11 output written                           97,632


In [42]:
na_counts = full.isna().sum()
print("NA count per output column (all 50):")
print(na_counts.to_string())

NA count per output column (all 50):
year                             0
combokey                         0
leaid                            0
schid                            0
lea_state                        0
sch_name                         0
lea_name                         0
jj                               0
seclusion_instances_wodis     6491
seclusion_instances_idea      6082
seclusion_instances_504       6252
seclusion_instances           6619
enrollment_total                11
enr_nonbinary                97632
enr_idea                        10
enr_504                         10
enr_el                          10
seclusion_rate                6619
sch_status_sped                  0
sch_status_charter               0
sch_status_alt                   0
sch_status_magnet                0
sch_virt_ind                 97632
grade_band                       0
fte_teachers                    21
fte_counselors                  21
fte_nurses                      21
fte_psych         

In [43]:
print("Flag counts (True):")
for col, expected in [("flag_missing_enrollment", 11), ("flag_small_enrollment", 1_212),
                      ("flag_missing_seclusion", 6_619), ("flag_fte_outlier", 68),
                      ("flag_analysis_ready", 91_013), ("flag_partial_enrollment", 0), ("flag_partial_idea", 0),
                      ("flag_partial_504", 0), ("flag_partial_el", 0), ("flag_missing_module", 0),
                      ("flag_missing_lea", 0)]:
    compare(col, int(full[col].sum()), expected)
print("grade_band 'other' (expected at least 1,921):", int((full["grade_band"] == "other").sum()),
      "| grade_band NA (expected 0):", int(full["grade_band"].isna().sum()))
print("JJ:", vc(full["jj"]), "| analysis-ready by JJ:", vc(analysis["jj"]))

Flag counts (True):
  flag_missing_enrollment                              expected       11   actual       11   match
  flag_small_enrollment                                expected    1,212   actual    1,212   match
  flag_missing_seclusion                               expected    6,619   actual    6,619   match
  flag_fte_outlier                                     expected       68   actual       68   match
  flag_analysis_ready                                  expected   91,013   actual   91,013   match
  flag_partial_enrollment                              expected        0   actual        0   match
  flag_partial_idea                                    expected        0   actual        0   match
  flag_partial_504                                     expected        0   actual        0   match
  flag_partial_el                                      expected        0   actual        0   match
  flag_missing_module                                  expected        0   actual        

In [44]:
print("quality_detail by issue (expected: 22,835 reserved_code, 510 non_digit_id, 232 subgroup_above_total, 71 fte_outlier):")
print(quality.groupby(["issue", "source_file"]).size().rename("rows").to_string())
print()
print("reserved_code rows by field and code:")
reserved_rows = quality[quality["issue"] == "reserved_code"]
print(reserved_rows.groupby(["raw_column", "raw_value"]).size().unstack(fill_value=0).to_string())
print("fte_outlier rows at", quality.loc[quality["issue"] == "fte_outlier", "combokey"].nunique(), "distinct schools")

quality_detail by issue (expected: 22,835 reserved_code, 510 non_digit_id, 232 subgroup_above_total, 71 fte_outlier):
issue                 source_file                
fte_outlier           School Support.csv                71
non_digit_id          School Characteristics.csv       510
reserved_code         Enrollment.csv                    82
                      LEA Characteristics.csv            1
                      Restraint and Seclusion.csv    18825
                      School Support.csv              3927
subgroup_above_total  Enrollment.csv                   232

reserved_code rows by field and code:
raw_value                    -11   -5  -5.00    -6  -6.00    -9  -9.00
raw_column                                                            
LEA_ENR                        0    1      0     0      0     0      0
SCH_ENR_504_F                  0   10      0     0      0     0      0
SCH_ENR_504_M                  0   10      0     0      0     0      0
SCH_ENR_IDEA_F           

In [45]:
print("Five sample keys (with lengths):")
sample_keys = full.iloc[[0, 24_000, 48_000, 72_000, N_ROSTER - 1]][["combokey", "leaid", "schid", "lea_state", "sch_name"]]
print(sample_keys.assign(len_combokey=sample_keys["combokey"].str.len(), len_leaid=sample_keys["leaid"].str.len(),
                         len_schid=sample_keys["schid"].str.len()).to_string(index=False))

Five sample keys (with lengths):
    combokey   leaid schid lea_state                           sch_name  len_combokey  len_leaid  len_schid
010000201705 0100002 01705        AL      Wallace Sch - Mt Meigs Campus            12          7          5
130228003990 1302280 03990        GA        Main Street Charter Academy            12          7          5
280366001011 2803660 01011        MS       South Pontotoc Middle School            12          7          5
420201000371 4202010 00371        PA               Clarks Summit El Sch            12          7          5
720003002085 7200030 02085        PR JUAN QUIRINDONGO MORELL (SUPERIOR)            12          7          5


## Step 12 - Validation of the written files (playbook section 6)
The three CSVs are read back from disk the way a teammate would read them (IDs as text:
`dtype={"combokey": str, "leaid": str, "schid": str}`) and checked against the single-year items of
the playbook section 6 checklist. Every check is an `assert`; the table at the end lists them all.
Floats are read with `float_precision="round_trip"`: pandas' default reader can change the last digit
of some long decimals (about 1 part in 10^16), so exact comparisons need the round-trip reader.

In [46]:
ID_DTYPES = {"combokey": str, "leaid": str, "schid": str}
re_full = pd.read_csv(OUT_FULL, dtype=ID_DTYPES, float_precision="round_trip")
re_analysis = pd.read_csv(OUT_ANALYSIS, dtype=ID_DTYPES, float_precision="round_trip")
re_default = pd.read_csv(OUT_FULL, dtype=ID_DTYPES)   # pandas' default float reader, for comparison
re_quality = pd.read_csv(OUT_QUALITY, dtype=str, keep_default_na=False)
CHECKS = []


def check(label, ok):
    CHECKS.append((label, "PASS" if ok else "FAIL"))
    assert ok, f"Validation failed: {label}"


print("reloaded:", re_full.shape, re_analysis.shape, re_quality.shape)

reloaded: (97632, 50) (91013, 50) (23648, 11)


In [47]:
# Keys, row counts, schema
check("full file: one row per roster school (97,632), in roster order",
      len(re_full) == N_ROSTER and (re_full["combokey"].to_numpy() == c["combokey"].to_numpy()).all())
check("exact 50-column schema and order in both school files (playbook sec. 3.1)",
      list(re_full.columns) == OUTPUT_COLUMNS and list(re_analysis.columns) == OUTPUT_COLUMNS)
check("IDs nonblank with lengths 12 / 7 / 5",
      re_full["combokey"].str.len().eq(12).all() and re_full["leaid"].str.len().eq(7).all()
      and re_full["schid"].str.len().eq(5).all())
letter = ~re_full["leaid"].str.fullmatch(r"\d{7}")
check("IDs are digits except the 510 documented letter LEAIDs; schid always 5 digits",
      int(letter.sum()) == 510 and re_full.loc[letter, "leaid"].str.fullmatch(r"[0-9A-Z]{7}").all()
      and re_full["schid"].str.fullmatch(r"\d{5}").all() and re_full.loc[~letter, "combokey"].str.fullmatch(r"\d{12}").all())
check("combokey == leaid + schid on every row", (re_full["combokey"] == re_full["leaid"] + re_full["schid"]).all())
leading_zero = int(re_full["combokey"].str.startswith("0").sum())
check(f"leading zeros kept on reload ({leading_zero:,} combokeys start with 0)",
      leading_zero > 0 and leading_zero == int(full["combokey"].str.startswith("0").sum()))
check("no duplicate school keys; (year, combokey) unique",
      re_full["combokey"].is_unique and not re_full.duplicated(["year", "combokey"]).any())
check("no duplicate district keys in the LEA file", lea["leaid"].is_unique)
check("every school matched a row in all modules and in the LEA file",
      not re_full["flag_missing_module"].any() and not re_full["flag_missing_lea"].any())

In [48]:
# Reload fidelity and types
reloaded = re_full.copy()
reloaded[TEXT_COLS] = reloaded[TEXT_COLS].astype(str)
reloaded[CATEGORY_OUT] = reloaded[CATEGORY_OUT].astype("string")
reloaded[NUMBER_COLS] = reloaded[NUMBER_COLS].astype("Float64")
expected = full.reset_index(drop=True)
non_numeric = TEXT_COLS + CATEGORY_OUT + FLAG_COLS


def as_float(frame, col):
    return frame[col].to_numpy(dtype=float, na_value=np.nan)


check("reloaded file equals the in-memory table: text, categories and flags identical",
      reloaded[non_numeric].equals(expected[non_numeric]))
check("reloaded file equals the in-memory table: every number bit-identical, same NA cells",
      all(np.array_equal(as_float(reloaded, col), as_float(expected, col), equal_nan=True) for col in NUMBER_COLS))
n_last_digit = sum(int((~np.isclose(as_float(re_default, col), as_float(expected, col), rtol=0, atol=0,
                                     equal_nan=True)).sum()) for col in NUMBER_COLS)
print(f"pandas' default reader changes the last digit of {n_last_digit:,} numeric values (all within 1e-12 relative)")
check(f"pandas' default reader: same NA cells, numbers within 1e-12 relative ({n_last_digit:,} values differ in the last digit)",
      all(np.allclose(as_float(re_default, col), as_float(expected, col), rtol=1e-12, atol=0, equal_nan=True)
          and (re_default[col].isna() == expected[col].isna()).all() for col in NUMBER_COLS))
check("flags reload as bool with no missing values", all(re_full[col].dtype == bool for col in FLAG_COLS))
check("text columns (IDs, names, state, year) have no missing values", re_full[TEXT_COLS].notna().all().all())
check("year is '2017-18' on every row", (re_full["year"] == YEAR).all())

pandas' default reader changes the last digit of 130,528 numeric values (all within 1e-12 relative)


In [49]:
# Counts, FTEs, sums and rates
numbers = re_full[NUMBER_COLS].to_numpy(dtype=float)
check("no negative or infinite values in any numeric column",
      not np.isinf(numbers).any() and (numbers[~np.isnan(numbers)] >= 0).all())
COUNT_OUT = [*SECL_OUT, "seclusion_instances", "enrollment_total", "enr_idea", "enr_504", "enr_el",
             "lea_enrollment", "lea_schools"]
check("count columns hold whole numbers only", all((re_full[col].dropna() % 1 == 0).all() for col in COUNT_OUT))
raw_zero = summary["zero"]
check("reported zeros stay zeros (seclusion and FTE fields: same zero counts as the raw files)",
      all(int((re_full[out] == 0).sum()) == int(raw_zero[raw]) for raw, out in {**SECL_COLS, **FTE_COLS}.items()))
parts_ok = re_full[SECL_OUT].notna().all(axis=1)
check("seclusion_instances is NA exactly when a component is NA, else the sum",
      (re_full["seclusion_instances"].notna() == parts_ok).all()
      and (re_full.loc[parts_ok, "seclusion_instances"] == re_full.loc[parts_ok, SECL_OUT].sum(axis=1)).all())
for total, (m, f) in {"enrollment_total": ("tot_enr_m", "tot_enr_f"), "enr_idea": ("idea_m", "idea_f"),
                      "enr_504": ("s504_m", "s504_f"), "enr_el": ("el_m", "el_f")}.items():
    rebuilt = strict_mf_total(school[m], school[f]).reset_index(drop=True).astype(float)
    check(f"{total} = M + F only when both are valid", rebuilt.equals(re_full[total].astype(float)))
den_re = re_full["enrollment_total"]
no_den = den_re.isna() | (den_re <= 0)
check("every rate / percentage is NA when the denominator is missing or not positive",
      re_full.loc[no_den, RATE_COLS].isna().all().all())
for col, numerator in [("seclusion_rate", "seclusion_instances"), ("teachers_per_100", "fte_teachers"),
                       ("counselors_per_100", "fte_counselors")]:
    rebuilt = per_100(re_full[numerator].astype("Float64"), den_re.astype("Float64")).astype(float)
    check(f"{col} = 100 * {numerator} / enrollment_total (valid numerator, denominator > 0)",
          rebuilt.equals(re_full[col].astype(float)))
fte_re = re_full[["fte_nurses", "fte_psych", "fte_social"]].astype("Float64")
check("support_fte_per_100 is NA unless nurses, psychologists and social workers are all valid",
      re_full.loc[fte_re.isna().any(axis=1), "support_fte_per_100"].isna().all()
      and support_per_100(fte_re, den_re.astype("Float64")).astype(float).equals(re_full["support_fte_per_100"].astype(float)))
for pct_col, (sub_col, _) in SUBGROUPS.items():
    above = re_full[sub_col] > den_re
    check(f"{pct_col} <= 100 and NA where {sub_col} > enrollment_total ({int(above.sum())} rows)",
          (re_full[pct_col].dropna() <= 100).all() and re_full.loc[above, pct_col].isna().all())
print(f"seclusion_rate > 100 on {int((re_full['seclusion_rate'] > 100).sum())} schools (allowed by playbook sec. 4)")

seclusion_rate > 100 on 155 schools (allowed by playbook sec. 4)

In [50]:
# Flags, categories, grade band, FTE review
flag_partial = ["flag_partial_enrollment", "flag_partial_idea", "flag_partial_504", "flag_partial_el"]
check("all flag_partial_* are False; enr_nonbinary and sch_virt_ind are NA (not collected in 2017-18)",
      not re_full[flag_partial].any().any() and re_full["enr_nonbinary"].isna().all() and re_full["sch_virt_ind"].isna().all())
rebuilt_flags = outcome_flags(re_full["seclusion_instances"].astype("Float64"), den_re.astype("Float64"))
check("flag_missing_seclusion / _enrollment / _small_enrollment / _analysis_ready follow playbook sec. 4",
      rebuilt_flags.equals(re_full[rebuilt_flags.columns]))
check("jj and school-status columns are Yes / No only",
      all(re_full[col].isin(["Yes", "No"]).all() for col in ["jj", *STATUS_COLS.values()]))
check("one grade_band per school, from the allowed set, equal to the sec. 4 rule on the roster flags",
      re_full["grade_band"].isin(["elementary", "middle", "high", "span", "other"]).all()
      and (re_full["grade_band"].to_numpy() == grade_band(school).astype(str).to_numpy()).all())
fte_rows = re_quality[re_quality["issue"] == "fte_outlier"]
flagged = set(re_full.loc[re_full["flag_fte_outlier"], "combokey"])
check(f"every flagged FTE school has a quality-detail row and vice versa ({len(flagged)} schools, {len(fte_rows)} rows)",
      flagged == set(fte_rows["combokey"]))
values_out = re_full.set_index("combokey")
check("flagged FTE values are kept as reported (quality cleaned_value == output value)",
      all(float(r.cleaned_value) == values_out.at[r.combokey, FTE_COLS[r.raw_column]] for r in fte_rows.itertuples()))
check("every value over a review trigger is flagged",
      all(re_full.loc[(re_full[col] > cut).fillna(False), "flag_fte_outlier"].all() for col, cut in FTE_TRIGGER.items()))

In [51]:
# Analysis file and quality detail
check("analysis file == full file filtered to flag_analysis_ready",
      re_analysis.equals(re_full[re_full["flag_analysis_ready"]].reset_index(drop=True)))
check("analysis file: valid seclusion total and positive enrollment on every row (91,013 rows)",
      len(re_analysis) == 91_013 and re_analysis["seclusion_instances"].notna().all()
      and (re_analysis["enrollment_total"] > 0).all())
check("quality detail has the sec. 3.3 columns in order", list(re_quality.columns) == QD_COLUMNS)
reserved_re = re_quality[re_quality["issue"] == "reserved_code"]
check(f"quality detail keeps the original reserved codes ({len(reserved_re):,} rows, all negative, e.g. '-9.00')",
      len(reserved_re) == 22_835 and (pd.to_numeric(reserved_re["raw_value"]) < 0).all()
      and reserved_re["raw_value"].isin(["-9.00"]).any() and reserved_re["cleaned_value"].eq("").all())
check("quality detail source_row is a positive record number on every row",
      re_quality["source_row"].str.fullmatch(r"[1-9]\d*").all())

pd.DataFrame(CHECKS, columns=["playbook sec. 6 check", "result"])

,playbook sec. 6 check,result
0,"full file: one row per roster school (97,632), in roster order",PASS
1,exact 50-column schema and order in both school files (playbook sec. 3.1),PASS
2,IDs nonblank with lengths 12 / 7 / 5,PASS
3,IDs are digits except the 510 documented letter LEAIDs; schid always 5 digits,PASS
4,combokey == leaid + schid on every row,PASS
5,"leading zeros kept on reload (18,227 combokeys start with 0)",PASS
6,"no duplicate school keys; (year, combokey) unique",PASS
7,no duplicate district keys in the LEA file,PASS
8,every school matched a row in all modules and in the LEA file,PASS
9,"reloaded file equals the in-memory table: text, categories and flags identical",PASS


In [52]:
print("Counts for the QA report:")
print("  by JJ:", vc(re_full["jj"]), "| small enrollment:", int(re_full["flag_small_enrollment"].sum()),
      "| analysis-ready:", int(re_full["flag_analysis_ready"].sum()))
print(pd.crosstab(re_full["jj"], [re_full["flag_analysis_ready"], re_full["flag_small_enrollment"]],
                  rownames=["jj"], colnames=["analysis_ready", "small"]).to_string())
print("Partial denominators (flag_partial_* True):", int(re_full[flag_partial].any(axis=1).sum()))
na_analysis = re_analysis.isna().sum()
print("Missing predictors in the analysis file (columns with NA):")
print(na_analysis[na_analysis > 0].to_string())

Counts for the QA report:
  by JJ: {'No': 97030, 'Yes': 602} | small enrollment: 1212 | analysis-ready: 91013
analysis_ready False        True       
small          False True   False True 
jj                                     
No              6408   164  89521   937
Yes               40     7    451   104
Partial denominators (flag_partial_* True): 0
Missing predictors in the analysis file (columns with NA):
enr_nonbinary          91013
sch_virt_ind           91013
fte_teachers               8
fte_counselors             8
fte_nurses                 8
fte_psych                  8
fte_social                 8
fte_leo                 1603
fte_guards              2150
counselors_per_100         8
teachers_per_100           8
support_fte_per_100        8
pct_idea                 183
pct_el                    10


### Hand-checks
First, real schools for each case named in playbook section 6. Then made-up rows for cases that never occur
in 2017-18 (zero enrollment, an unknown grade flag) and a few edge cases. The made-up rows go through
the **same functions** used above (`strict_mf_total`, `seclusion_total`, `per_100`, `subgroup_pct`,
`support_per_100`, `outcome_flags`, `grade_band`) and are never added to the outputs.

In [53]:
def first_school(mask):
    return school.loc[mask, "combokey"].iloc[0]


yes = school[GRADE_COLS].eq("Yes")
REAL_CASES = {
    "missing M and F (-5, Peabody MA)": "250936001493",
    "missing M and F (-6, force certified)": "390439399999",
    "missing support FTE (nurses reserved)": first_school(school["fte_nurses"].isna() & school["enrollment_total"].notna()),
    "KG-8 -> span": first_school(yes[GRADE_E + GRADE_M].all(axis=1) & ~yes[GRADE_H].any(axis=1)),
    "KG-12 -> span": first_school(yes.all(axis=1)),
    "high only -> high": first_school(yes[GRADE_H].all(axis=1) & ~yes[GRADE_E + GRADE_M].any(axis=1)),
    "flagged FTE outlier (Del Valle HS guards)": "481662001424",
}
cols = ["sch_name", "enrollment_total", "seclusion_instances", "seclusion_rate", "pct_idea", "fte_nurses",
        "support_fte_per_100", "fte_guards", "grade_band", "flag_missing_enrollment", "flag_fte_outlier",
        "flag_analysis_ready"]
real = values_out.loc[list(REAL_CASES.values()), cols].assign(case=list(REAL_CASES)).set_index("case", append=True)
real

,,sch_name,enrollment_total,seclusion_instances,seclusion_rate,pct_idea,fte_nurses,support_fte_per_100,fte_guards,grade_band,flag_missing_enrollment,flag_fte_outlier,flag_analysis_ready
combokey,case,,,,,,,,,,,,
250936001493,"missing M and F (-5, Peabody MA)",Center,NaN,NaN,NaN,NaN,NaN,NaN,NaN,elementary,True,False,False
390439399999,"missing M and F (-6, force certified)",Preble County Educational Service Center Alternative School,NaN,NaN,NaN,NaN,NaN,NaN,0.0,span,True,False,False
362058000102,missing support FTE (nurses reserved),Passages Academy / Bronx Hope,77.0,0.0,0.0,0.000000,NaN,NaN,NaN,span,False,False,True
010027000062,KG-8 -> span,Perdido Elementary School,472.0,0.0,0.0,15.254237,0.0,0.000000,0.0,span,False,False,True
010000901403,KG-12 -> span,Alabama School For Deaf,206.0,0.0,0.0,NaN,9.0,5.339806,4.0,span,False,False,True
010000500871,high only -> high,Albertville High School,1449.0,0.0,0.0,5.244997,1.0,0.069013,0.0,high,False,False,True
481662001424,flagged FTE outlier (Del Valle HS guards),DEL VALLE H S,3138.0,0.0,0.0,11.822817,1.0,0.031867,102579.0,high,False,True,True


In [54]:
r = values_out.loc
for key in ["250936001493", "390439399999"]:
    assert pd.isna(r[key, "enrollment_total"]) and pd.isna(r[key, "seclusion_rate"]) and pd.isna(r[key, "pct_idea"])
    assert r[key, "flag_missing_enrollment"] and not r[key, "flag_analysis_ready"]
key = REAL_CASES["missing support FTE (nurses reserved)"]
assert pd.isna(r[key, "support_fte_per_100"]) and not pd.isna(r[key, "enrollment_total"])
assert r[REAL_CASES["KG-8 -> span"], "grade_band"] == "span"
assert r[REAL_CASES["KG-12 -> span"], "grade_band"] == "span"
assert r[REAL_CASES["high only -> high"], "grade_band"] == "high"
assert r["481662001424", "fte_guards"] == 102_579 and r["481662001424", "flag_fte_outlier"]
print("Real-school hand-checks passed.")
q_del_valle = re_quality[(re_quality["combokey"] == "481662001424") & (re_quality["issue"] == "fte_outlier")]
print(q_del_valle[["raw_column", "raw_value", "action", "cleaned_value", "evidence"]].to_string(index=False))

Real-school hand-checks passed.
         raw_column raw_value                            action cleaned_value                                                                                                                                                                                                                                           evidence
SCH_FTESECURITY_GUA 102579.00 retained; flag_fte_outlier = True      102579.0 Group trigger: review trigger fte_guards > 200 (playbook sec. 1 step 10); value 102579; enrollment_total 3138; fte_teachers 200.5; peer max 34204 (other schools in LEA: 12); OCR Appndx C STAF: Check 84, Univariate STAF; value kept as reported


In [55]:
NA = pd.NA
fake = pd.DataFrame({
    "case": ["only F missing", "zero enrollment", "all valid", "one seclusion part missing",
             "missing support FTE", "subgroup above total", "small school"],
    "m":    [10, 0, 60, 60, 60, 60, 3],
    "f":    [NA, 0, 40, 40, 40, 40, 4],
    "seclusion_instances_wodis": [1, 0, 2, 2, 2, 2, 0],
    "seclusion_instances_idea":  [0, 0, 3, NA, 3, 3, 0],
    "seclusion_instances_504":   [0, 0, 0, 0, 0, 0, 0],
    "fte_nurses": [1, 1, 1, 1, NA, 1, 1], "fte_psych": [0.5] * 7, "fte_social": [0.5] * 7,
    "idea": [1, 0, 12, 12, 12, 120, 1],
}).set_index("case")
fake = fake.astype("Float64")
fake["enrollment_total"] = strict_mf_total(fake["m"], fake["f"])
fake["seclusion_instances"] = seclusion_total(fake)
fake["seclusion_rate"] = per_100(fake["seclusion_instances"], fake["enrollment_total"])
fake["support_fte_per_100"] = support_per_100(fake, fake["enrollment_total"])
fake["pct_idea"] = subgroup_pct(fake["idea"], fake["enrollment_total"])
fake = fake.join(outcome_flags(fake["seclusion_instances"], fake["enrollment_total"]))
result_cols = ["enrollment_total", "seclusion_instances", "seclusion_rate", "support_fte_per_100", "pct_idea",
               "flag_missing_seclusion", "flag_missing_enrollment", "flag_small_enrollment", "flag_analysis_ready"]
EXPECTED_FAKE = pd.DataFrame({
    "enrollment_total":    [NA, 0, 100, 100, 100, 100, 7],
    "seclusion_instances": [1, 0, 5, NA, 5, 5, 0],
    "seclusion_rate":      [NA, NA, 5.0, NA, 5.0, 5.0, 0.0],
    "support_fte_per_100": [NA, NA, 2.0, 2.0, NA, 2.0, 2 / 7 * 100],
    "pct_idea":            [NA, NA, 12.0, 12.0, 12.0, NA, 1 / 7 * 100],
    "flag_missing_seclusion":  [False, False, False, True, False, False, False],
    "flag_missing_enrollment": [True, True, False, False, False, False, False],
    "flag_small_enrollment":   [False, True, False, False, False, False, True],
    "flag_analysis_ready":     [False, False, True, False, True, True, True],
}, index=fake.index)
for col in result_cols[:5]:
    EXPECTED_FAKE[col] = EXPECTED_FAKE[col].astype("Float64")
pd.testing.assert_frame_equal(fake[result_cols], EXPECTED_FAKE, check_exact=False, rtol=1e-12)
print("Made-up enrollment / seclusion / rate rows behave as expected (zero enrollment -> rates NA,",
      "flag_missing_enrollment True, not analysis-ready).")
fake[result_cols]

Made-up enrollment / seclusion / rate rows behave as expected (zero enrollment -> rates NA, flag_missing_enrollment True, not analysis-ready).


,enrollment_total,seclusion_instances,seclusion_rate,support_fte_per_100,pct_idea,flag_missing_seclusion,flag_missing_enrollment,flag_small_enrollment,flag_analysis_ready
case,,,,,,,,,
only F missing,<NA>,1.0,<NA>,<NA>,<NA>,False,True,False,False
zero enrollment,0.0,0.0,<NA>,<NA>,<NA>,False,True,True,False
all valid,100.0,5.0,5.0,2.0,12.0,False,False,False,True
one seclusion part missing,100.0,<NA>,<NA>,2.0,12.0,True,False,False,False
missing support FTE,100.0,5.0,5.0,<NA>,12.0,False,False,False,True
subgroup above total,100.0,5.0,5.0,2.0,<NA>,False,False,False,True
small school,7.0,0.0,0.0,28.571429,14.285714,False,False,True,True


In [56]:
def grade_row(yes_grades, unknown=()):
    row = {g: ("Yes" if g in yes_grades else "No") for g in GRADE_COLS}
    row.update({g: NA for g in unknown})
    return row


GRADE_CASES = {
    "unknown grade flag (G03 missing)": (grade_row(GRADE_E, unknown=["SCH_GRADE_G03"]), NA),
    "KG-8": (grade_row(GRADE_E + GRADE_M), "span"),
    "KG-12": (grade_row(GRADE_COLS), "span"),
    "G06-G12": (grade_row(GRADE_M + GRADE_H), "span"),
    "high only (G09-G12)": (grade_row(GRADE_H), "high"),
    "middle only (G06-G08)": (grade_row(GRADE_M), "middle"),
    "elementary only (KG-G05)": (grade_row(GRADE_E), "elementary"),
    "no KG-G12 grade (preschool or ungraded only)": (grade_row([]), "other"),
}
fake_grades = pd.DataFrame([row for row, _ in GRADE_CASES.values()], index=list(GRADE_CASES)).astype("string")
bands = grade_band(fake_grades)
expected_bands = pd.Series([band for _, band in GRADE_CASES.values()], index=list(GRADE_CASES), dtype="string")
pd.testing.assert_series_equal(bands, expected_bands, check_names=False)
print("Made-up grade rows behave as expected (unknown flag -> NA).")
pd.DataFrame({"grade_band": bands})

Made-up grade rows behave as expected (unknown flag -> NA).


,grade_band
unknown grade flag (G03 missing),<NA>
KG-8,span
KG-12,span
G06-G12,span
high only (G09-G12),high
middle only (G06-G08),middle
elementary only (KG-G05),elementary
no KG-G12 grade (preschool or ungraded only),other
